# Per-region LMM, permutation, and enrichment: whole-brain cFos atlas

Fits one model per brain region (`log_density ~ genotype * sex * light`), extracts
every contrast of interest as a weighted sum over the 8 design-cell means, then
tests whether effects are *broad* (Section 11) and whether they are *concentrated
in particular anatomy* (Section 12).

### Pipeline

| | |
|---|---|
| 1-2 | Config; load cleaned long-format density + mouse metadata |
| 3 | Contrast machinery -- weight vectors over the 8 design cells |
| 4-5 | Fit every region, extract every contrast, BH-FDR correct |
| 6 | Volcano plots |
| 7 | Diagnostics: p-histograms, convergence, SE, global gain, heteroscedasticity |
| 8 | QQ diagnostic: "many weak" vs "few strong" |
| 9 | Region coverage by Allen CCF major structure |
| 10 | Permutation engine: exact label permutation + Freedman-Lane |
| 11 | Global breadth test |
| 12 | Enrichment on pre-specified region sets |
| 12x | **NEW** Enrichment on every Allen major subdivision |
| 13 | Design/power audit, gain audit, readout |

## What changed in this version

**1. Sign convention is now CONTROL-POSITIVE everywhere.** `code_map` already had
`cre/+ = +1`, but the four genotype *simple* effects were coded the other way
(`cre/cre - cre/+`) and named `MKO_vs_control`, so one output table carried both
directions at once. Those four are now `control_vs_MKO` and are coded
`cre/+ - cre/cre`, matching the interactions, the main effects, the PLSC notebook
and the global LMM.

> **This flips the sign of the four genotype simple effects** relative to earlier
> runs. Nothing else in this notebook changes -- the interactions and main
> effects were already control-positive. `PERM_CONTRASTS` and the enrichment
> section were renamed to match.

**2. The enrichment statistic is now subset-versus-whole-brain, not
subset-versus-complement.** `mean_z_vs_all = mean(z | in S) - mean(z | all regions)`
replaces `mean_z_diff = mean(z | in S) - mean(z | out of S)` as the default.

> **This changes no p-value.** Algebraically
> `mean_z_vs_all = (1 - k/N) * mean_z_diff`, and `(1 - k/N)` is a positive
> constant applied identically to the observed statistic and every null draw --
> both nulls hold `k` and `N` fixed (size-matched draws in the region null;
> restricted label shuffles preserve cell counts exactly, so gating is
> permutation-invariant in the label null). Permutation `p` and `z_vs_null` come
> out identical; only the reported effect size rescales. Section 12b asserts this
> numerically. It is the better number to *report* -- it reads as "this set sits
> X above the brain-wide average" and it stops inflating effect sizes for large
> sets -- but it is not a re-analysis and must not be described as one.

**3. Enrichment is now run on every Allen major subdivision** (Section 12x), as a
separate BH family from the pre-specified sets. The **random** region null is
deliberately not reported for subdivisions: a subdivision *is* a contiguous
anatomical block, so the random null would call almost all of them enriched for
reasons that have nothing to do with the biology. Subdivisions occupying a large
share of the brain are flagged, because at `k/N` near 0.5 neither null has
resolution.

**4. Fixed: `enrichment_region_null` did not intersect the requested set with
regions having a usable observed statistic.** Null draws are all-usable by
construction, so the observed statistic could be computed over fewer effective
regions than the null it was compared against.

### Known open items

- `USE_BATCH_RANDOM_EFFECT` must be `False` for Sections 10-13; the closed-form
  fast path assumes plain OLS. Asserted, not assumed.
- SEs/p-values use a z-test on the fixed effects (no Satterthwaite/KR df
  correction, unlike `emmeans`/`lmerTest` in R) -- spot-check a few regions
  against the R pipeline if exact agreement matters for the write-up.

## 1. Config and imports

In [ ]:
import numpy as np, pandas as pd, patsy, warnings, difflib
from pathlib import Path
from itertools import product
from math import comb
from dataclasses import dataclass, field
from scipy.stats import norm, levene
from scipy.linalg import null_space
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Paths -- repo-relative, so this notebook runs wherever the repository is
# checked out. PROJECT_ROOT is found by walking up from the working directory
# until a folder containing `data/` appears. If you keep the data outside the
# repository, set PROJECT_ROOT by hand and everything below follows.
# ---------------------------------------------------------------------------
def find_project_root(markers=("data", "notebooks")):
    """Nearest ancestor directory containing all of `markers`. Requiring both
    `data/` and `notebooks/` means a stray `data/` folder somewhere on the path
    cannot be mistaken for the repository root."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if all((candidate / m).is_dir() for m in markers):
            return candidate
    raise RuntimeError(
        f"No ancestor of {here} contains {list(markers)}. Set PROJECT_ROOT by hand."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR     = PROJECT_ROOT / "data"
RESULTS_DIR  = PROJECT_ROOT / "results"

LONG_CSV = DATA_DIR / "compiled" / "cleaned_output" / "density_by_region_leaves_long.csv"
INFO_CSV = DATA_DIR / "mouse_info_template.csv"

# ### FIXED ### OUT_CSV used to land on the lab share while OUT_DIR was a
# relative folder, so the contrast table and the figures built from it ended up
# in two different places. Both now go to results/per_region/.
OUT_DIR = RESULTS_DIR / "per_region"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "regional_contrasts_whole_brain.csv"

# ---- Minimum-data thresholds ----
# ### CHOICE ###
# Two separate thresholds doing two separate jobs, plus a floor on total
# region sample size:
#   MIN_REGION_N        -- a region needs at least this many total observations
#                          (across all cells) before ANY contrast is attempted.
#   MIN_CELL_N          -- hard floor per design cell. A contrast is only
#                          attempted if every cell it touches has at least this
#                          many mice. Below this, the contrast is SKIPPED for
#                          that region entirely (not just flagged).
#   UNDERPOWERED_CELL_N -- soft flag. At or above MIN_CELL_N but below this, the
#                          contrast IS computed and reported, but flagged
#                          `underpowered=True` so it can be excluded from the
#                          primary BH-FDR family rather than silently diluting
#                          it with noisy estimates.
#
# ### KNOWN LIMITATION ###
# UNDERPOWERED_CELL_N = 3 flags a contrast only when a cell it touches holds
# fewer than 3 mice. The realised design cells here are 3-8 animals, so on this
# dataset the soft flag fires rarely and `q_powered` is close to `q_all` for most
# contrasts. That is the intended behaviour at this threshold, not an oversight:
# the hard floor MIN_CELL_N = 1 is what gates a contrast out entirely (338 of 360
# regions clear it), and UNDERPOWERED_CELL_N marks the thinnest survivors rather
# than re-gating the analysis.
#
# If the threshold is ever raised, note that the condition is `n < UNDERPOWERED_CELL_N`,
# so a value of 5 does NOT flag a cell of exactly 5. Set it one above the cell
# size you want caught.
MIN_REGION_N = 3
MIN_CELL_N = 1
UNDERPOWERED_CELL_N = 3

# ---- Batch structure toggle ----
# ### CHOICE ###
# True  -> smf.mixedlm(..., groups=batch): batch (immuno batch -- staining
#          noise, NOT the sex-nested perfusion batch used in the PLSC
#          permutation scheme) as a random intercept.
# False -> smf.ols(...): no batch term at all. Useful as a direct diagnostic
#          comparison -- if per-region SEs are dramatically tighter with this
#          off, the random-intercept variance component is likely unstable at
#          this per-region sample size (see Section 7b).
#
# ### REQUIRED BY SECTIONS 10-12 ###
# The permutation engine's fast path assumes a saturated OLS cell-means model,
# where each fitted cell value IS that cell's sample mean. Random-intercept
# shrinkage breaks that equivalence. Sections 10-12 assert this is False.
USE_BATCH_RANDOM_EFFECT = False

## 2. Load and merge data

In [ ]:
long = pd.read_csv(LONG_CSV)     # long: Acronym, Brain Region, Level_0..9, mouse_id, density
info = pd.read_csv(INFO_CSV)     # id, genotype, sex, light, batch, litter, age_days
info["id"] = info["id"].astype(str)
long["mouse_id"] = long["mouse_id"].astype(str)

d = long.merge(info, left_on="mouse_id", right_on="id", how="left")

# Log-transform with a half-minimum-positive-value pseudocount, matching the
# convention used throughout this project (PLSC pipeline, R LMM pipeline).
pseudo = d.loc[d["density"] > 0, "density"].min() / 2
d["log_density"] = np.log(d["density"] + pseudo)

# ### CHOICE ###
# Fix category order explicitly (rather than relying on alphabetical/observed
# order) so the dummy coding -- and therefore the SIGN of every contrast
# (L-D, cre/cre-cre/+, M-F) -- is known and consistent, not inferred.
d["genotype"] = pd.Categorical(d["genotype"], categories=["cre/+", "cre/cre"])
d["sex"]      = pd.Categorical(d["sex"],      categories=["F", "M"])
d["light"]    = pd.Categorical(d["light"],    categories=["D", "L"])
d["batch"]    = d["batch"].astype(str)   # immuno batch -- staining noise only

print(f"{d['mouse_id'].nunique()} mice, {d['Acronym'].nunique()} regions, {len(d)} region-mouse rows")

## 3. Contrast machinery

Every contrast -- simple effect, two-way (pooled or split), three-way, or main
effect -- is computed the same way: a length-8 weight vector over the 8
`(genotype, sex, light)` design cells, projected through this region's own
fitted cell-design matrix and coefficients. Unifying all contrast types onto one
mechanism guarantees an identical convention rather than risking drift between,
say, a patsy-row-difference method for simple effects and a separately-derived
weight-vector method for interactions.

### Sign convention: CONTROL-POSITIVE

`cre/+ = +1`, `M = +1`, `L = +1`, throughout. A positive `geno:light` estimate
means the light effect is more positive (or less negative) in `cre/+` than in
`cre/cre`. A positive `control_vs_MKO` estimate means higher cFos in control.

This is the same convention as the PLSC notebook and the global LMM, and the
three sets of weight vectors are built by the same construction so they cannot
drift.

> **Changed in this version.** The four genotype simple effects used to be coded
> `cre/cre - cre/+` and named `MKO_vs_control`, while every interaction and main
> effect was already control-positive. That put both directions in one table.
> They are now `control_vs_MKO`, coded `cre/+ - cre/cre`. **Their sign is
> negated relative to earlier runs.** `light_vs_dark` is unaffected; it always
> matched `code_map`.

### The contrast set

Twenty-five contrasts are built, and the eighteen the manuscript reports are
kept. See the `PAPER_CONTRASTS` block in the next code cell for which are
dropped and why.

| type | n | what |
|---|---|---|
| simple | 12 | every two-cell comparison: each factor varied at each of the 4 combinations of the other two |
| two-way, pooled | 3 | `geno:sex`, `geno:light`, `sex:light`, each averaged over the omitted factor |
| two-way, split | 6 | each two-way *within* each level of the third factor |
| three-way | 1 | `geno:sex:light` |
| main effect | 3 | each factor, averaged over both others |

### CHOICE -- pooled vs split two-ways, and why both are here
A pooled two-way divides by 2 to average over the omitted factor's two levels,
keeping the estimate on the same "difference of cell-mean averages" log scale as
everything else. A split two-way does not divide -- it is fully saturated within
its stratum.

The split set is complete and symmetric: `geno:light` within each sex,
`geno:sex` within each light condition, `sex:light` within each genotype. That
symmetry is deliberate. A pooled two-way plus a significant three-way is
ambiguous on its own -- the pooled term is an average of two strata that the
three-way says are different -- so the split terms are the decomposition that
makes the three-way readable. Note that this is a **planned decomposition** of
the highest-order term in a fully crossed design, not a fishing expedition: the
three-way is the gate, and each split pair sums to twice the pooled term and
differs by exactly the three-way, by construction.

`sex:light | control` and `sex:light | MKO` are the pair that states the
crossover directly: they are "the sex difference in light response, within each
genotype". They carry no genotype term and are therefore **unchanged** by the
sign fix above.

In [ ]:
CELLS = list(product(["cre/+", "cre/cre"], ["F", "M"], ["D", "L"]))
FACTORS = ("genotype", "sex", "light")
code_map = {"cre/+": 1, "cre/cre": -1, "F": -1, "M": 1, "D": -1, "L": 1}

# Display labels, so contrast names read as English rather than as coding levels
PRETTY = {"cre/+": "control", "cre/cre": "MKO", "L": "light", "D": "dark", "M": "M", "F": "F"}

TERM_FACTORS = {
    "geno:sex":       ("genotype", "sex"),
    "geno:light":     ("genotype", "light"),
    "sex:light":      ("sex", "light"),
    "geno:sex:light": ("genotype", "sex", "light"),
}


def cell_dict(cell):
    return dict(zip(FACTORS, cell))


def simple_effect_weight(fix: dict, vary_col: str, ref_val: str, target_val: str):
    """8-vector: +1 at the target cell, -1 at the reference cell, 0 elsewhere.
    A simple effect compares exactly two specific cells -- no averaging."""
    w = []
    for cell in CELLS:
        cd = cell_dict(cell)
        if not all(cd[k] == v for k, v in fix.items()):
            w.append(0.0)
        elif cd[vary_col] == target_val:
            w.append(1.0)
        elif cd[vary_col] == ref_val:
            w.append(-1.0)
        else:
            w.append(0.0)
    return np.array(w)


def interaction_weight(term: str, split: dict = None):
    """8-vector for a pooled or split interaction contrast.

    term  : key of TERM_FACTORS
    split : None for the pooled version, or e.g. {"sex": "M"} / {"light": "L"} /
            {"genotype": "cre/+"} to restrict to one level of the third factor.

    Pooled terms divide by 2^(3 - order) to average over the omitted factors,
    keeping the estimate on the same 'difference of cell-mean averages' log
    scale as everything else. A split term is fully saturated within its
    stratum and is NOT divided -- so `split_L + split_D = 2 * pooled`, and
    `split_L - split_D` is exactly the three-way. Those two identities are
    asserted below.
    """
    facs = TERM_FACTORS[term]
    if split is not None and set(split) & set(facs):
        raise ValueError(
            f"Cannot split '{term}' by {list(split)} -- that factor is already one of "
            f"the interacting terms. 'The geno:sex interaction within males' is not a "
            f"coherent sub-question the way 'the geno:light interaction within males' is."
        )
    w = []
    for cell in CELLS:
        cd = cell_dict(cell)
        if split is not None and any(cd[k] != v for k, v in split.items()):
            w.append(0.0)
            continue
        wi = float(np.prod([code_map[cd[f]] for f in facs]))
        if split is None:
            wi /= 2 ** (3 - len(facs))
        w.append(wi)
    return np.array(w)


def main_effect_weight(factor: str):
    """8-vector for a pooled main effect (e.g. light, averaged across genotype
    and sex). Divided by 4 (2x2 = the omitted factors' levels) for the same
    reason the pooled two-ways divide by 2."""
    return np.array([code_map[cell_dict(c)[factor]] / 4 for c in CELLS])

In [ ]:
# ---- 12 simple effects: every factor varied at every combination of the others ----
# spec: (varying factor, reference level, target level, name stem, display order
#        of the two fixed factors)
# ### CHANGED -- control-positive ###
# The genotype simple effect is now cre/+ MINUS cre/cre, so it agrees with
# code_map (cre/+ = +1) and therefore with every interaction and main effect in
# this notebook. Previously it ran the other way while keeping the name
# "MKO_vs_control", which put two sign conventions in one output table. The NAME
# changed with the arithmetic on purpose -- a silent flip would be worse.
SIMPLE_SPECS = [
    ("light",    "D",       "L",     "light_vs_dark",  ("genotype", "sex")),
    ("genotype", "cre/cre", "cre/+", "control_vs_MKO", ("light", "sex")),
    ("sex",      "F",       "M",     "M_vs_F",         ("genotype", "light")),
]

SIMPLE_CONTRASTS = {}
for vary, ref, tgt, stem, show in SIMPLE_SPECS:
    fixed_factors = [f for f in FACTORS if f != vary]
    levels = {"genotype": ["cre/+", "cre/cre"], "sex": ["F", "M"], "light": ["D", "L"]}
    for a in levels[show[0]]:
        for b in levels[show[1]]:
            fix = {show[0]: a, show[1]: b}
            name = f"{stem} | {PRETTY[a]} | {PRETTY[b]}"
            SIMPLE_CONTRASTS[name] = simple_effect_weight(fix, vary, ref, tgt)

# ---- 3 pooled two-ways + the three-way ----
INTERACTION_CONTRASTS = {
    "geno:sex":       interaction_weight("geno:sex"),
    "geno:light":     interaction_weight("geno:light"),
    "sex:light":      interaction_weight("sex:light"),
    "geno:sex:light": interaction_weight("geno:sex:light"),
}

# ---- 6 split two-ways: each two-way within each level of the third factor ----
SPLIT_SPECS = [
    ("geno:light", "sex",      ["M", "F"]),
    ("geno:sex",   "light",    ["L", "D"]),
    ("sex:light",  "genotype", ["cre/+", "cre/cre"]),
]
for term, by, levels in SPLIT_SPECS:
    for lv in levels:
        INTERACTION_CONTRASTS[f"{term} | {PRETTY[lv]}"] = interaction_weight(term, split={by: lv})

# ---- 3 pooled main effects ----
# Needed for claims like "region X responds to light, averaged across genotype
# and sex" -- NOT provided by the simple effects, which are each specific to one
# genotype x sex cell. A significant main effect plus non-significant
# interactions is NOT proof the effect is "independent of sex/genotype"; a
# non-significant interaction may simply reflect insufficient power. An
# equivalence test (e.g. TOST) against a pre-specified negligible-effect
# threshold is the rigorous way to make that a positive claim.
MAIN_EFFECT_CONTRASTS = {
    "light (main effect)":    main_effect_weight("light"),
    "genotype (main effect)": main_effect_weight("genotype"),
    "sex (main effect)":      main_effect_weight("sex"),
}

ALL_CONTRASTS = {
    **{k: ("simple", v)      for k, v in SIMPLE_CONTRASTS.items()},
    **{k: ("interaction", v) for k, v in INTERACTION_CONTRASTS.items()},
    **{k: ("main_effect", v) for k, v in MAIN_EFFECT_CONTRASTS.items()},
}

# ---- Structural assertions on the split/pooled relationship ----
# These are not decoration: if the coding convention ever drifts, the split
# terms stop being a decomposition of the three-way and every claim built on
# them silently changes meaning.
for term, by, levels in SPLIT_SPECS:
    a = INTERACTION_CONTRASTS[f"{term} | {PRETTY[levels[0]]}"]
    b = INTERACTION_CONTRASTS[f"{term} | {PRETTY[levels[1]]}"]
    assert np.allclose((a + b) / 2, INTERACTION_CONTRASTS[term]), \
        f"{term}: split levels do not average to the pooled term"
    three_way = INTERACTION_CONTRASTS["geno:sex:light"]
    assert np.allclose(a - b, three_way) or np.allclose(b - a, three_way), \
        f"{term}: difference of split levels is not the three-way"

# ---- Restrict to the contrasts the manuscript reports ----------------------
# ### CHOICE ###
# All 25 contrasts are CONSTRUCTED above because the structural assertions
# directly above need the pooled two-ways and both levels of every split pair to
# verify the coding convention. Only the reported ones are carried forward.
#
# Dropping a contrast here cannot move any kept contrast's numbers: BH-FDR is
# applied WITHIN each contrast across regions (Section 5), never across
# contrasts, and every contrast is a weight vector evaluated independently
# against the same per-region fit.
#
# Dropped (7): the 3 pooled two-ways, which are superseded by their split forms
# (Figure 2D reports the splits), and the 4 `M_vs_F` simple effects, which no
# figure reports per-region.
#
# All three split PAIRS are kept, including `geno:sex | light` / `geno:sex | dark`.
# Figure 4 reports the dark level and Figure 2D currently omits the pair, but the
# two levels are a single decomposition of the three-way -- their difference IS
# the three-way and their average is the pooled geno:sex term -- so reporting one
# without the other leaves the decomposition asymmetric.
PAPER_CONTRASTS = [
    # main effects -- Figure 1 (light), Figure S7 (genotype), Figure 2B (sex)
    "light (main effect)", "genotype (main effect)", "sex (main effect)",
    # omnibus three-way -- Figures 2B, 3, 6
    "geno:sex:light",
    # split two-ways -- Figure 2D; the geno:sex pair carries Figure 4
    "geno:light | M", "geno:light | F",
    "sex:light | control", "sex:light | MKO",
    "geno:sex | light", "geno:sex | dark",
    # simple effects of light -- Figure 2E
    "light_vs_dark | control | M", "light_vs_dark | control | F",
    "light_vs_dark | MKO | M",     "light_vs_dark | MKO | F",
    # simple effects of genotype -- Figures 2F, 4G
    "control_vs_MKO | dark | M",  "control_vs_MKO | dark | F",
    "control_vs_MKO | light | M", "control_vs_MKO | light | F",
]

_missing = [c for c in PAPER_CONTRASTS if c not in ALL_CONTRASTS]
assert not _missing, f"PAPER_CONTRASTS names were not built above: {_missing}"
_n_built = len(ALL_CONTRASTS)
ALL_CONTRASTS = {k: ALL_CONTRASTS[k] for k in PAPER_CONTRASTS}

# The three component dicts are filtered to match. Later cells iterate them
# (the volcano grids in Section 6, the QQ panels in Section 8, the Freedman-Lane
# reduced-model listing in Section 10d) and then index ALL_CONTRASTS, so leaving
# them holding all 25 makes those cells raise a KeyError on the first dropped
# name. They must stay in step with ALL_CONTRASTS.
SIMPLE_CONTRASTS      = {k: v for k, v in SIMPLE_CONTRASTS.items()      if k in ALL_CONTRASTS}
INTERACTION_CONTRASTS = {k: v for k, v in INTERACTION_CONTRASTS.items() if k in ALL_CONTRASTS}
MAIN_EFFECT_CONTRASTS = {k: v for k, v in MAIN_EFFECT_CONTRASTS.items() if k in ALL_CONTRASTS}
assert (len(SIMPLE_CONTRASTS) + len(INTERACTION_CONTRASTS)
        + len(MAIN_EFFECT_CONTRASTS)) == len(ALL_CONTRASTS), \
    "component dicts are out of step with ALL_CONTRASTS"

print(f"{len(ALL_CONTRASTS)} contrasts reported (of {_n_built} constructed):")
for _group in ("main_effect", "interaction", "simple"):
    print(f"\n{_group}:")
    for _k, (_ct, _) in ALL_CONTRASTS.items():
        if _ct == _group:
            print(f"  {_k}")

# ---- Sign-convention assertions -------------------------------------------
# Every contrast with a NET genotype loading must load POSITIVELY on cre/+ cells.
# This is what silently failed before the control-positive flip: the interactions
# were control-positive and the genotype simple effects were not.
#
# ### FIXED ###
# The previous version applied this test to every contrast whose NAME contained
# "geno", which swept in the interactions -- and an interaction has ZERO net
# loading on any single factor level by construction (the three-way puts
# +1, -1, -1, +1 across the four cre/+ cells, summing to 0). The test therefore
# asserted something false of every interaction and the cell raised on the first
# one it reached. Interactions are covered instead by the structural assertions
# above and by the explicit three-way weight check below.
_NET_LOADING_CONTRASTS = [nm for nm in ALL_CONTRASTS
                          if "control_vs_MKO" in nm or nm == "genotype (main effect)"]
for _nm in _NET_LOADING_CONTRASTS:
    _w = ALL_CONTRASTS[_nm][1]
    _pos = sum(_w[i] for i, c in enumerate(CELLS) if c[0] == "cre/+")
    assert _pos > 0, f"'{_nm}' is not control-positive (cre/+ loading {_pos:+.2f})"

# Interactions: net loading must be exactly zero, which is the property the test
# above cannot express. A non-zero value would mean a contrast is contaminated
# with a lower-order effect.
for _nm, (_ct, _w) in ALL_CONTRASTS.items():
    if _ct != "interaction":
        continue
    _pos = sum(_w[i] for i, c in enumerate(CELLS) if c[0] == "cre/+")
    assert np.isclose(_pos, 0), \
        f"'{_nm}' has a net cre/+ loading of {_pos:+.2f}; an interaction must be 0"

_i = CELLS.index(("cre/+", "M", "L"))
assert np.isclose(INTERACTION_CONTRASTS["geno:sex:light"][_i], 1.0), \
    "three-way weight at (cre/+, M, L) must be +1 under the control-positive convention"

print("\nSign convention: CONTROL-POSITIVE (cre/+ = +1, M = +1, L = +1) -- asserted.")
print("  Genotype simple effects are `control_vs_MKO`; their sign is NEGATED")
print("  relative to any run before this one. Interactions/main effects unchanged.")

## 4. Core fitting function

In [ ]:
def build_cell_design_matrix(design_info):
    """8 x k matrix -- one row per CELLS entry, in the same coefficient space
    (k = number of fixed-effect parameters) as this region's fitted model. Uses
    the model's own `design_info` so dummy coding is guaranteed consistent with
    however the model was actually fit."""
    rows = []
    for g, s, l in CELLS:
        row = pd.DataFrame([{"genotype": g, "sex": s, "light": l}])
        row["genotype"] = pd.Categorical(row["genotype"], categories=d["genotype"].cat.categories)
        row["sex"]      = pd.Categorical(row["sex"],      categories=d["sex"].cat.categories)
        row["light"]    = pd.Categorical(row["light"],    categories=d["light"].cat.categories)
        dm = patsy.build_design_matrices([design_info], row, return_type="dataframe")[0]
        rows.append(dm.iloc[0].values)
    return np.array(rows)


def region_cell_n(sub, g, s, l):
    return len(sub[(sub.genotype == g) & (sub.sex == s) & (sub.light == l)])


def fit_one_region(acronym, dat):
    """Fit one region's model, then extract every contrast in ALL_CONTRASTS.

        cvec = weight_vector @ cell_design_matrix
        est  = cvec @ beta
        se   = sqrt(cvec @ cov(beta) @ cvec)
        z, p = est / se, two-sided normal p-value

    ### CHOICE ###
    Inference uses a z-test on the fixed effects (matching how MixedLM has been
    used throughout this project, e.g. the whole-brain global_cfos model), NOT
    Satterthwaite/KR degrees-of-freedom t-tests the way emmeans/lmerTest do in
    R. At this n the difference is usually small -- but if the R and Python
    pipelines' p-values ever need to match closely, this is the first place to
    look.
    """
    sub = dat[(dat["Acronym"] == acronym) & dat.log_density.notna()]
    if len(sub) < MIN_REGION_N:
        return None

    try:
        if USE_BATCH_RANDOM_EFFECT:
            fitted = smf.mixedlm("log_density ~ genotype * sex * light",
                                 sub, groups=sub["batch"]).fit(method="lbfgs", maxiter=200)
            converged = fitted.converged
            k = len(fitted.fe_params)
            beta   = fitted.fe_params.values
            cov_fe = fitted.cov_params().iloc[:k, :k].values
            design_info = fitted.model.data.design_info
        else:
            fitted = smf.ols("log_density ~ genotype * sex * light", sub).fit()
            converged = True   # OLS is closed-form -- no iterative optimizer to fail
            beta   = fitted.params.values
            cov_fe = fitted.cov_params().values
            design_info = fitted.model.data.design_info
    except Exception:
        return None

    M = build_cell_design_matrix(design_info)
    cell_n = {(g, s, l): region_cell_n(sub, g, s, l) for g, s, l in CELLS}

    results = []
    for cname, (ctype, w) in ALL_CONTRASTS.items():
        involved_ns = [cell_n[c] for c, wi in zip(CELLS, w) if wi != 0]
        if not involved_ns or min(involved_ns) < MIN_CELL_N:
            continue   # hard floor -- contrast not attempted for this region

        cvec = w @ M
        est  = cvec @ beta
        se   = np.sqrt(cvec @ cov_fe @ cvec)
        z    = est / se
        # ### FIX ###
        # norm.sf, not 1 - norm.cdf: the latter underflows to EXACTLY 0 once
        # |z| exceeds about 8.3, which then makes -log10(p) infinite and
        # silently poisons every aggregate statistic in Sections 11-12.
        p    = 2 * norm.sf(abs(z))

        results.append({
            "Acronym": acronym, "contrast": cname, "contrast_type": ctype,
            "log_estimate": est, "se": se, "z": z, "p": p,
            "fold_change": np.exp(est), "converged": converged,
            "min_cell_n": min(involved_ns),
            "underpowered": min(involved_ns) < UNDERPOWERED_CELL_N,
        })
    return pd.DataFrame(results) if results else None

## 5. Run across all regions, then BH-FDR correct

In [ ]:
acronyms = d["Acronym"].unique()
print(f"Fitting {len(acronyms)} regions x {len(ALL_CONTRASTS)} contrasts  |  "
      f"batch random effect: {USE_BATCH_RANDOM_EFFECT}  |  "
      f"MIN_CELL_N={MIN_CELL_N}  UNDERPOWERED_CELL_N={UNDERPOWERED_CELL_N}")

chunks, n_failed_fit = [], 0
for i, a in enumerate(acronyms):
    res = fit_one_region(a, d)
    if res is None:
        n_failed_fit += 1
    else:
        chunks.append(res)
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(acronyms)} regions processed")

all_results = pd.concat(chunks, ignore_index=True)
print(f"\nGot {len(all_results)} region-contrast rows across "
      f"{all_results['Acronym'].nunique()} regions ({n_failed_fit} regions skipped entirely)")

n_nonconverged = (~all_results["converged"]).sum()
n_underpowered = all_results["underpowered"].sum()
print(f"{n_nonconverged} rows non-converged ({n_nonconverged/len(all_results):.1%})")
print(f"{n_underpowered} rows flagged underpowered ({n_underpowered/len(all_results):.1%})")

In [ ]:
# Drop non-converged rows before FDR -- an unreliable p-value from a
# non-converged fit can shift the correction threshold for every OTHER
# (reliable) row in that contrast's family, the same contamination risk as
# leaving underpowered rows in.
n_before = len(all_results)
all_results = all_results[all_results["converged"]].copy()
print(f"Dropped {n_before - len(all_results)} non-converged rows before FDR "
      f"({(n_before - len(all_results))/max(n_before,1):.1%})")

# ### CHOICE ###
# BH-FDR is computed TWICE per contrast:
#   q_all     -- on every converged row (what you'd get doing nothing further)
#   q_powered -- restricted to well-powered rows only (underpowered=False)
# Underpowered rows contribute noisy p-values that can shift the correction
# threshold for everyone else in that contrast's family. q_powered is NaN for
# underpowered rows by construction -- it is the number to trust for any
# headline claim; q_all is kept as a transparency/sensitivity check, not a
# substitute.
#
# ### FLAG -- multiplicity across contrasts ###
# Correcting WITHIN each contrast's family means the contrasts are separate
# separate families. That is defensible under the hierarchical framing (the
# three-way is the confirmatory gate; the rest are planned decompositions), but
# it is a claim you are making, not a neutral default. State it in the methods.
all_results["q_all"] = np.nan
for cname, sub in all_results.groupby("contrast"):
    _, qvals, _, _ = multipletests(sub["p"], method="fdr_bh")
    all_results.loc[sub.index, "q_all"] = qvals

all_results["q_powered"] = np.nan
powered = all_results[~all_results["underpowered"]]
for cname, sub in powered.groupby("contrast"):
    _, qvals, _, _ = multipletests(sub["p"], method="fdr_bh")
    all_results.loc[sub.index, "q_powered"] = qvals

all_results.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV}")

In [ ]:
# ### CHOICE ###
# Build the summary from ALL_CONTRASTS (every contrast the pipeline is SUPPOSED
# to test), not from all_results.groupby(...) directly -- a plain groupby
# silently omits any contrast that ended up with zero surviving rows (e.g. every
# region gated out by MIN_CELL_N), which would look identical to that contrast
# simply not existing rather than having failed entirely.
contrast_type_lookup = {cname: ctype for cname, (ctype, _) in ALL_CONTRASTS.items()}

summary = (all_results.groupby(["contrast_type", "contrast"])
           .agg(n_tested=("q_all", "size"),
                n_underpowered=("underpowered", "sum"),
                n_sig_all=("q_all", lambda x: (x < 0.05).sum()),
                n_sig_powered_only=("q_powered", lambda x: (x < 0.05).sum()))
           .reset_index()
           .set_index("contrast"))

full_index = pd.DataFrame({
    "contrast": list(ALL_CONTRASTS.keys()),
    "contrast_type": [contrast_type_lookup[c] for c in ALL_CONTRASTS.keys()],
}).set_index("contrast")

summary_full = (full_index.join(summary.drop(columns="contrast_type"))
                .fillna({"n_tested": 0, "n_underpowered": 0, "n_sig_all": 0,
                         "n_sig_powered_only": 0})
                .astype({"n_tested": int, "n_underpowered": int, "n_sig_all": int,
                         "n_sig_powered_only": int})
                .reset_index()[["contrast_type", "contrast", "n_tested", "n_underpowered",
                                "n_sig_all", "n_sig_powered_only"]])

# Order by definition order within type, not alphabetically (which would
# scramble e.g. the 4 light_vs_dark rows away from each other).
defn_order = {c: i for i, c in enumerate(ALL_CONTRASTS)}
type_order = {"simple": 0, "interaction": 1, "main_effect": 2}
summary_full["_t"] = summary_full["contrast_type"].map(type_order)
summary_full["_d"] = summary_full["contrast"].map(defn_order)
summary_full = summary_full.sort_values(["_t", "_d"]).drop(columns=["_t", "_d"])

print(summary_full.to_string(index=False))

## 6. Volcano plots

In [ ]:
df = pd.read_csv(OUT_CSV)
df["log2FC"] = np.log2(df["fold_change"])

# q_powered is NaN for underpowered rows by construction -- fall back to q_all
# for THOSE POINTS ONLY so they still plot, but keep them visually distinct
# (open/gray markers) so a viewer cannot mistake an underpowered point for a
# properly-corrected one. Only q_powered < 0.05 counts as "sig" (red); an
# underpowered point is never colored red, regardless of q_all.
df["q_display"]   = df["q_powered"].fillna(df["q_all"])
df["neg_log10_q"] = -np.log10(df["q_display"])


def plot_volcano_grid(contrast_order, title, fname, ncols=4):
    """Auto-grids to fit however many contrasts are passed -- with many contrasts
    the old fixed nrows/ncols arguments silently dropped panels off the end."""
    n = len(contrast_order)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 4.2 * nrows),
                             sharex=True, sharey=True, squeeze=False)
    axes_flat = axes.reshape(-1)

    for ax, cname in zip(axes_flat, contrast_order):
        sub = df[df.contrast == cname]
        sig          = sub["q_powered"] < 0.05
        underpowered = sub["underpowered"]

        ax.scatter(sub.log2FC[underpowered], sub.neg_log10_q[underpowered],
                   facecolors="none", edgecolors="lightgray", s=14, linewidths=0.6,
                   label="underpowered")
        ax.scatter(sub.log2FC[~underpowered & ~sig], sub.neg_log10_q[~underpowered & ~sig],
                   c="lightgray", s=12, alpha=0.6, label="n.s.")
        ax.scatter(sub.log2FC[sig], sub.neg_log10_q[sig], c="#C44E52", s=16,
                   label="q_powered<0.05")
        ax.axhline(-np.log10(0.05), color="gray", lw=0.5, ls="--")
        ax.axvline(0, color="gray", lw=0.5)
        ax.set_title(f"{cname}\n(n={sub['q_display'].notna().sum()}, sig={sig.sum()})",
                     fontsize=8)

        for _, r in sub[sig].nsmallest(5, "q_powered").iterrows():
            ax.annotate(r["Acronym"], (r.log2FC, r.neg_log10_q), fontsize=6, alpha=0.8)

    for ax in axes_flat[n:]:
        ax.axis("off")
    for ax in axes[-1, :]:
        ax.set_xlabel("log2(fold change)")
    for ax in axes[:, 0]:
        ax.set_ylabel("-log10(q)")
    axes_flat[0].legend(fontsize=7, loc="upper left")
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(OUT_DIR / fname, dpi=200)
    plt.show()


plot_volcano_grid(list(SIMPLE_CONTRASTS),
                  f"Simple effects ({len(SIMPLE_CONTRASTS)})", "volcano_simple.png")
plot_volcano_grid(list(INTERACTION_CONTRASTS),
                  f"Interactions ({len(INTERACTION_CONTRASTS)})", "volcano_interactions.png")
plot_volcano_grid(list(MAIN_EFFECT_CONTRASTS),
                  f"Main effects ({len(MAIN_EFFECT_CONTRASTS)})",
                  "volcano_main_effects.png", ncols=3)

## 7. Diagnostics

These cells validate pipeline health -- they do not change any result, but they
are what tell you whether the p-values and SEs above are trustworthy. Worth
re-running whenever `USE_BATCH_RANDOM_EFFECT`, `MIN_CELL_N`, or the underlying
dataset changes.

### 7a. p-value histograms, per contrast

In [ ]:
# Under the null, p-values should be roughly uniform (flat histogram). A spike
# near 0 with an otherwise-flat tail is what real, BH-survivable signal looks
# like. A histogram elevated broadly across low-to-mid p -- or worse, RISING
# toward p=1 -- usually signals miscalibration (e.g. inflated SEs from a
# thin/degenerate cell) rather than either real signal or a clean null.
all_contrast_names = list(ALL_CONTRASTS.keys())
n_contrasts = len(all_contrast_names)
ncols = 5
nrows = int(np.ceil(n_contrasts / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 2.8 * nrows),
                         sharex=True, squeeze=False)
axes_flat = axes.reshape(-1)
for ax, cname in zip(axes_flat, all_contrast_names):
    sub = all_results[all_results.contrast == cname]
    ax.hist(sub["p"], bins=20, range=(0, 1), color="#4C72B0", edgecolor="white")
    ax.set_title(cname, fontsize=7)
for ax in axes_flat[n_contrasts:]:
    ax.axis("off")
plt.tight_layout(); plt.show()

### 7b. Convergence rate and SE distribution, per contrast

In [ ]:
# A rising-toward-1 p-histogram for one specific contrast often comes from a
# subset of absurdly large SEs (near-singular batch variance, quasi-separation
# in a thin cell) dragging p toward 1 regardless of the point estimate -- check
# whether convergence and SE spread are uniform across contrasts, or whether one
# is disproportionately unstable.
print("Convergence rate by contrast (should be ~uniform; a low outlier means")
print("that contrast is riding disproportionately on bad fits):")
print(all_results.groupby("contrast")["converged"].mean().to_string())

fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 2.8 * nrows), squeeze=False)
axes_flat = axes.reshape(-1)
for ax, cname in zip(axes_flat, all_contrast_names):
    sub = all_results[all_results.contrast == cname]
    ax.hist(sub["se"], bins=30, color="#55A868")
    ax.axvline(sub["se"].median(), color="red", lw=1)
    ax.set_title(cname, fontsize=7)
for ax in axes_flat[n_contrasts:]:
    ax.axis("off")
plt.tight_layout(); plt.show()

print("\nSE distribution by contrast:")
print(all_results.groupby("contrast")["se"].describe().to_string())

### 7c. Global-gain check

In [ ]:
# A per-mouse brain-wide gain effect (a brain that stains/images uniformly
# hotter than others) inflates apparent regional effects in the SAME direction
# across nearly every region simultaneously, because it is a property of the
# mouse, not the region -- this can masquerade as a strong, consistent "light
# effect" (or any other effect) in a volcano plot with a skewed,
# mostly-one-directional point cloud. This pipeline does NOT correct for it
# (unlike the PLSC pipeline's explicit per-mouse centering, Xc = Xa -
# Xa.mean(axis=1)) -- check this directly before trusting a strongly
# one-directional volcano plot.
mouse_level = d.groupby("mouse_id")["log_density"].mean().reset_index()
mouse_level = mouse_level.merge(info.rename(columns={"id": "mouse_id"}), on="mouse_id")

print("Per-mouse whole-brain mean log-density, control males, by light condition:")
print(mouse_level[(mouse_level.genotype == "cre/+") & (mouse_level.sex == "M")]
      .sort_values("light")[["mouse_id", "light", "log_density"]].to_string(index=False))

# ### CHOICE ###
# If this shows a consistent light/dark gap that is large relative to
# within-group spread -- especially if driven by 1-2 animals in a thin cell --
# the fix (not applied here) is either:
#   (a) add each mouse's whole-brain mean log_density as a covariate, or
#   (b) center each mouse's log_density by their own whole-brain mean before
#       fitting (the PLSC pipeline's Xc step).
# Either changes what the contrasts MEAN -- testing regional PATTERN net of
# overall brain-wide level, not raw magnitude -- a deliberate interpretive
# shift, not a drop-in fix. Note that Section 11's global breadth test is
# sensitive to exactly this, so read the two together.

### 7d. Residual heteroscedasticity by sex

Checks whether the **pooled** model (fitting both sexes together, extracting
per-sex contrasts afterward) is defensible, or whether residual variance differs
enough by sex that a sex-split model would be more honest despite the power cost.

In [ ]:
# Refit each region (same spec as fit_one_region) but keep raw residuals and
# fitted values tagged by sex, rather than the contrast outputs -- checking a
# model ASSUMPTION (homoscedasticity across sex), not re-deriving results.
resid_rows = []
for acronym in d["Acronym"].unique():
    sub = d[(d["Acronym"] == acronym) & d.log_density.notna()]
    if len(sub) < MIN_REGION_N:
        continue
    try:
        if USE_BATCH_RANDOM_EFFECT:
            fitted = smf.mixedlm("log_density ~ genotype * sex * light",
                                 sub, groups=sub["batch"]).fit(method="lbfgs", maxiter=200)
            if not fitted.converged:
                continue
        else:
            fitted = smf.ols("log_density ~ genotype * sex * light", sub).fit()
    except Exception:
        continue

    resid_rows.append(pd.DataFrame({
        "Acronym": acronym, "sex": sub["sex"].values, "genotype": sub["genotype"].values,
        "light": sub["light"].values, "fitted": fitted.fittedvalues.values,
        "resid": fitted.resid.values,
    }))

resid_df = pd.concat(resid_rows, ignore_index=True)
print(f"Collected residuals from {resid_df['Acronym'].nunique()} regions, "
      f"{len(resid_df)} total observations")

In [ ]:
# Pooled check: residual spread by sex, aggregated across all regions.
# NOTE: with thousands of pooled observations, Levene's test here has enough
# power to flag even trivial variance differences as "significant" -- read the
# EFFECT SIZE (variance ratio), not just the p-value, and treat this as
# preliminary; the per-region check below is the one that distinguishes a real
# widespread pattern from a pooled-test power artifact.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for sx, color in [("M", "#4C72B0"), ("F", "#C44E52")]:
    sub = resid_df[resid_df.sex == sx]
    axes[0].scatter(sub.fitted, sub.resid, s=4, alpha=0.25, color=color, label=sx)
axes[0].axhline(0, color="gray", lw=0.5)
axes[0].set(xlabel="fitted value", ylabel="residual", title="Residuals vs. fitted, by sex")
axes[0].legend()

resid_df["abs_resid"] = resid_df["resid"].abs()
axes[1].boxplot([resid_df.loc[resid_df.sex == "M", "abs_resid"],
                 resid_df.loc[resid_df.sex == "F", "abs_resid"]],
                tick_labels=["M", "F"], showfliers=False)
axes[1].set(ylabel="|residual|", title="Residual magnitude by sex (pooled across regions)")
plt.tight_layout(); plt.show()

m_resid = resid_df.loc[resid_df.sex == "M", "resid"]
f_resid = resid_df.loc[resid_df.sex == "F", "resid"]
stat, p = levene(m_resid, f_resid)
print(f"Pooled Levene's test (M vs F residual variance): stat={stat:.3f}, p={p:.4f}")
print(f"  M: var={m_resid.var():.4f}  n={len(m_resid)}   F: var={f_resid.var():.4f}  n={len(f_resid)}")
print(f"  Variance ratio (M/F): {m_resid.var() / f_resid.var():.3f}")

In [ ]:
# Per-region check -- the one that actually matters. Under the null (no real sex
# heteroscedasticity anywhere), ~5% of regions should come back significant at
# p<0.05 by chance alone, with a roughly flat p-value histogram. A rate well
# above 5%, or a histogram piled up near 0, indicates real widespread
# heteroscedasticity worth switching to sex-split models for -- a handful of
# individual outlier regions (rate near 5%, but a few extreme variance ratios)
# is NOT the same finding and does not by itself justify abandoning the pooled
# model.
region_levene = []
for acronym, sub in resid_df.groupby("Acronym"):
    m = sub.loc[sub.sex == "M", "resid"]
    f = sub.loc[sub.sex == "F", "resid"]
    if len(m) < 3 or len(f) < 3:
        continue
    stat, p = levene(m, f)
    region_levene.append({
        "Acronym": acronym, "levene_p": p,
        "var_ratio_M_over_F": m.var() / f.var() if f.var() > 0 else np.nan,
        "n_M": len(m), "n_F": len(f),
    })
region_levene = pd.DataFrame(region_levene)

frac_sig = (region_levene["levene_p"] < 0.05).mean()
print(f"{len(region_levene)} regions tested")
print(f"Fraction with per-region Levene p<0.05: {frac_sig:.1%}  (expect ~5% under the null)")
print(f"Median per-region variance ratio (M/F): "
      f"{region_levene['var_ratio_M_over_F'].median():.3f}  (1.0 = balanced)")

# Flag near-degenerate fits (a variance ratio of, e.g., 1e29 is not a real
# biological effect -- it means one sex's residuals collapsed to ~0 in that
# region, almost always from a saturated/near-perfect fit in a very thin cell).
degenerate = region_levene[region_levene["var_ratio_M_over_F"] > 10]
if len(degenerate):
    print(f"\n!! {len(degenerate)} region(s) show near-degenerate variance ratios "
          f"(likely a saturated fit, not a real effect) -- treat their contrast")
    print(f"   estimates with caution regardless of significance:")
    print(degenerate[["Acronym", "var_ratio_M_over_F", "n_M", "n_F"]].to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(region_levene["levene_p"], bins=20, range=(0, 1), color="#55A868", edgecolor="white")
ax[0].axhline(len(region_levene) / 20, color="red", ls="--", lw=1, label="expected under null")
ax[0].set(xlabel="per-region Levene p-value", ylabel="count",
          title="Should be ~flat if no heteroscedasticity")
ax[0].legend()

finite_ratios = region_levene.loc[np.isfinite(region_levene["var_ratio_M_over_F"]) &
                                  (region_levene["var_ratio_M_over_F"] > 0), "var_ratio_M_over_F"]
ax[1].hist(np.log2(finite_ratios[finite_ratios < 10]), bins=30, color="#4C72B0", edgecolor="white")
ax[1].axvline(0, color="gray", lw=1)
ax[1].set(xlabel="log2(variance ratio, M/F)", ylabel="count",
          title="Should center on 0 if balanced (degenerate outliers excluded)")
plt.tight_layout(); plt.show()

## 8. QQ diagnostic: "many weak" vs "few strong"

For each `light_vs_dark` simple contrast, sort the observed p-values and plot
`-log10(observed)` against `-log10(expected)` under a uniform null for that rank
(`i/(n+1)`). Under pure noise, points hug the diagonal. A broad,
individually-weak effect lifts the *whole cloud* off the diagonal; a handful of
strong regions plus noise instead spears a few points off the top while the bulk
still hugs the line. These look very different even when a naive per-region FDR
count treats them identically (zero survivors).

This is the visual version of the question Section 11 answers formally.

In [ ]:
def plot_qq(contrast_names, title, fname, ncols=4):
    n = len(contrast_names)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.4 * ncols, 4.2 * nrows),
                             sharex=True, sharey=True, squeeze=False)
    axes_flat = axes.reshape(-1)
    for ax, cname in zip(axes_flat, contrast_names):
        sub = all_results[all_results.contrast == cname].sort_values("p")
        n_r = len(sub)
        if n_r == 0:
            ax.axis("off"); continue
        expected = -np.log10(np.arange(1, n_r + 1) / (n_r + 1))
        observed = -np.log10(sub["p"].values)
        ax.scatter(expected, observed, s=10, alpha=0.5, color="#4C72B0")
        lim = max(expected.max(), observed.max()) * 1.05
        ax.plot([0, lim], [0, lim], color="gray", lw=1, ls="--")
        ax.set(xlim=(0, lim), ylim=(0, lim), xlabel="expected -log10(p)",
               title=f"{cname}\n(n={n_r})")
    for ax in axes_flat[n:]:
        ax.axis("off")
    for ax in axes[:, 0]:
        ax.set_ylabel("observed -log10(p)")
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(OUT_DIR / fname, dpi=200)
    plt.show()


plot_qq([c for c in SIMPLE_CONTRASTS if c.startswith("light_vs_dark")],
        "Light vs dark, by genotype x sex", "qq_light_vs_dark_by_group.png")

# The split sex:light pair is the crossover stated directly -- the sex
# difference in light response, within each genotype.
plot_qq(["sex:light | control", "sex:light | MKO", "geno:sex:light"],
        "Crossover contrasts", "qq_crossover.png")

## 9. Region coverage and significance, by Allen CCF major structure

Breaks the region list down by the standard Allen CCFv3 major divisions --
both how many regions were actually tested in each (coverage is uneven: thin,
small-nuclei-heavy divisions like HY or PAL often have more regions gated out by
`MIN_CELL_N` than Isocortex does), and how many cleared the q threshold.

### CHOICE
Major-division membership is looked up from the atlas structure tree
(`brainglobe_atlasapi`) rather than from the `Level_0..9` columns already present
in `long`/`d`. Those columns encode depth in the ontology, but which level number
corresponds to "major division" is not fixed across branches (Isocortex sits
deeper than fiber tracts), so a fixed `Level_N == major structure` assumption
would silently misclassify regions in some branches. Walking each region's actual
ancestor path is correct regardless of tree depth.

The `bg_atlas` object built here is reused in Section 12 for CCF centroids.

In [ ]:
from brainglobe_atlasapi import BrainGlobeAtlas

bg_atlas = BrainGlobeAtlas("allen_mouse_25um")   # matches the resolution used
                                                 # elsewhere in this project

# Canonical Allen CCFv3 major divisions. Order here is print order -- roughly
# rostral/cortical to caudal, then white matter / ventricles last.
MAJOR_STRUCTURES = [
    "Isocortex", "OLF", "HPF", "CTXsp",   # cortex-related
    "STR", "PAL",                          # cerebral nuclei
    "TH", "HY",                            # interbrain
    "MB",                                  # midbrain
    "P", "MY",                             # hindbrain
    "CB",                                  # cerebellum
    "fiber tracts", "VS",                  # white matter / ventricular system
]


def major_structure(acronym):
    """First ancestor of `acronym` that is one of MAJOR_STRUCTURES, or None.

    ### FIX ###
    `bg_atlas.structures` supports item lookup by acronym, but its `.keys()`
    only enumerates the underlying integer structure IDs -- so checking
    `acronym not in bg_atlas.structures.keys()` fails for EVERY acronym, valid
    or not. Use try/except on the lookup itself instead."""
    try:
        path_ids = bg_atlas.structures[acronym]["structure_id_path"]
    except KeyError:
        return None
    path_acronyms = [bg_atlas.structures[i]["acronym"] for i in path_ids]
    for major in MAJOR_STRUCTURES:
        if major in path_acronyms:
            return major
    return None


region_to_major = {a: major_structure(a) for a in all_results["Acronym"].unique()}

unmapped = sorted(a for a, m in region_to_major.items() if m is None)
if unmapped:
    print(f"WARNING: {len(unmapped)} / {len(region_to_major)} acronyms did not map to a "
          f"major structure (atlas-version/naming mismatch?) -- excluded from the "
          f"breakdown below. Spot-check a few against {bg_atlas.atlas_name}:")
    print(unmapped)
else:
    print(f"All {len(region_to_major)} tested acronyms mapped cleanly to a major structure.")

In [ ]:
def structure_breakdown(contrast_name, q_col="q_all", q_thresh=0.05):
    """Per Allen CCF major division: how many regions were tested for
    `contrast_name`, and how many cleared `q_col < q_thresh`. Regions that did
    not map to a major structure are excluded and reported, not silently
    dropped."""
    sub = all_results[all_results["contrast"] == contrast_name].copy()
    sub["major_structure"] = sub["Acronym"].map(region_to_major)
    n_excluded = sub["major_structure"].isna().sum()
    sub = sub[sub["major_structure"].notna()]

    out = (sub.groupby("major_structure")
              .agg(n_tested=("Acronym", "nunique"),
                   n_sig=(q_col, lambda x: (x < q_thresh).sum()))
              .reindex(MAJOR_STRUCTURES)   # fixed order; divisions with zero
              .fillna(0)                    # tested regions still show as 0
              .astype(int))
    out["pct_sig"] = (100 * out["n_sig"] / out["n_tested"].replace(0, np.nan)).round(1)

    print(f"Structure breakdown for contrast: {contrast_name}   "
          f"(significance: {q_col} < {q_thresh})")
    if n_excluded:
        print(f"({n_excluded} region-rows excluded -- unmapped acronym, see warning above)")
    print(out.to_string())
    print(f"\nTOTAL: {out['n_tested'].sum()} tested, {out['n_sig'].sum()} significant "
          f"({100*out['n_sig'].sum()/max(out['n_tested'].sum(),1):.1f}%)")
    return out


# ### CHOICE ### -- change to inspect any other contrast in ALL_CONTRASTS
CONTRAST_FOR_BREAKDOWN = "geno:sex | light"
_ = structure_breakdown(CONTRAST_FOR_BREAKDOWN, q_thresh=0.05)

## 10. Permutation engine

Per-region FDR asks "which individual regions survive correction for ~300
simultaneous tests". That is the wrong question for a broad, individually-modest
effect: a whole-brain shift can clear a pooled test long before any single region
clears a corrected per-region one. Sections 10-12 ask the aggregate questions
instead -- is the effect *broad* (11), and is it *concentrated in pre-specified
anatomy* (12).

**The permutation unit is always the mouse.** One shuffle per permutation,
propagated to every region before any p-value is recomputed. Shuffling
region-by-region independently would destroy the between-region correlation that
is genuinely present (nearby and connected regions co-vary), making the null far
too narrow. Shuffling once per permutation preserves that correlation structure
*by construction* -- which is exactly the property a naive
hypergeometric/Fisher enrichment test lacks, and the core of the Fulcher et al.
(2021) spatial-null critique.

**One run, many tests.** The engine stores the full `n_perm x n_region` matrix of
`-log10 p` **and** signed `z`. The expensive part happens once; global breadth,
any region subset, and any subset-vs-subset comparison are then cheap slices of
the same null, and are mutually consistent because they share it.

### 10a. Wide matrix, mouse design, and the fixed region set

The engine works off a single `regions x mice` matrix rather than the long
dataframe, so a permutation is a reshuffle of a length-`n_mice` vector -- no
groupby, no per-region slicing, no refitting.

In [ ]:
# ---- regions x mice matrix of log_density ----
wide = d.pivot_table(index="Acronym", columns="mouse_id", values="log_density", aggfunc="mean")

# ---- mouse design table, aligned column-for-column with `wide` ----
mouse_design = (d[["mouse_id", "genotype", "sex", "light"]]
                .drop_duplicates("mouse_id")
                .set_index("mouse_id")
                .reindex(wide.columns))
assert mouse_design.notna().all().all(), "A mouse in `wide` has no design row -- check the info merge."

CELL_INDEX = {cell: i for i, cell in enumerate(CELLS)}


def design_to_cell_idx(design_df):
    """(n_mice,) int array of 0..7 indices into CELLS, in `wide` column order."""
    return np.array([CELL_INDEX[(g, s, l)] for g, s, l in
                     zip(design_df["genotype"], design_df["sex"], design_df["light"])])


OBS_CELL_IDX = design_to_cell_idx(mouse_design)
cell_counts = np.bincount(OBS_CELL_IDX, minlength=8)
print("Observed design cells (genotype, sex, light) -> n mice:")
for cell, n in zip(CELLS, cell_counts):
    print(f"  {cell}: {n}")
assert (cell_counts > 0).all(), "An empty design cell -- the saturated model is not identified."

# ### CHOICE ###
# REQUIRE_COMPLETE: restrict the permutation region set to regions with a
# non-missing value for EVERY mouse.
#
# Why this is not just tidiness: if a region is missing a mouse, then WHICH
# design cell loses an animal depends on where that mouse's shuffled label
# lands. Per-cell n therefore becomes permutation-DEPENDENT, so (a) MIN_CELL_N
# gating can fire on some permutations and not others, making the number of
# regions contributing to the null statistic drift between permutations, and
# (b) the observed statistic is computed over a region set no permutation is
# guaranteed to match. Both push the same way: null sums come out too small,
# observed looks too extreme, permutation p is anticonservative.
#
# ### FIXED FROM PREVIOUS VERSION ###
# The old Section 8 did `ps = ps[~np.isnan(ps)]` inside the permutation loop
# while computing the observed statistic over all eligible regions -- exactly
# this bug.
#
# REQUIRE_COMPLETE=True is also a hard requirement for Freedman-Lane (10d): the
# reduced-model hat matrix is computed once and shared across regions, which is
# only valid if every region has the same design.
REQUIRE_COMPLETE = True

complete_mask = wide.notna().all(axis=1)
if REQUIRE_COMPLETE:
    PERM_REGIONS = wide.index[complete_mask].values
    print(f"\n{len(PERM_REGIONS)} / {len(wide)} regions have complete data across all "
          f"{wide.shape[1]} mice and will be used for permutation")
    dropped = wide.index[~complete_mask].values
    if len(dropped):
        print(f"  {len(dropped)} regions dropped for missingness: {list(dropped[:15])}"
              f"{' ...' if len(dropped) > 15 else ''}")
else:
    PERM_REGIONS = wide.index.values
    print(f"\nREQUIRE_COMPLETE=False -- using all {len(PERM_REGIONS)} regions; per-cell n is "
          f"permutation-dependent for {(~complete_mask).sum()} of them, and Freedman-Lane "
          f"is unavailable.")

XMAT = wide.loc[PERM_REGIONS].values          # (n_regions, n_mice), may contain NaN
REGION_POS = {a: i for i, a in enumerate(PERM_REGIONS)}
print(f"XMAT: {XMAT.shape[0]} regions x {XMAT.shape[1]} mice")

### 10b. Vectorized closed-form contrast

`genotype * sex * light` is a fully saturated 2x2x2 factorial (8 cells, 8
parameters), so under plain OLS each fitted cell value **is** that cell's sample
mean. Any contrast therefore has

```
est = w . cell_means
var = pooled_var * sum_i (w_i^2 / n_i)      pooled_var = (sum within-cell SSR) / (N - 8)
```

with no design matrix, no inversion, no `patsy` call -- and it vectorizes across
all regions at once. That is what makes thousands of permutations across ~300
regions an interactive operation rather than an overnight job. The equivalence
breaks under `USE_BATCH_RANDOM_EFFECT=True`, hence the assertion.

In [ ]:
assert not USE_BATCH_RANDOM_EFFECT, (
    "Sections 10-12 assume plain OLS (saturated cell-means model). With "
    "USE_BATCH_RANDOM_EFFECT=True the closed form no longer holds and every "
    "permutation would need a real smf.mixedlm refit."
)


def contrast_fast(X, cell_idx, w, min_cell_n=MIN_CELL_N):
    """Closed-form contrast for every region at once.

    X        : (n_regions, n_mice) log_density, NaN allowed
    cell_idx : (n_mice,) ints 0..7, index into CELLS
    w        : (8,) contrast weight vector -- the same vectors as ALL_CONTRASTS

    Returns est, se, z, p, neglog10p -- each (n_regions,). z, p and neglog10p
    are NaN wherever the region is gated out (a cell this contrast touches is
    below min_cell_n, zero residual df, or a degenerate SE).

    ### FIX ###
    `neglog10p` is computed from `norm.logsf` rather than as `-log10(p)`.
    `2 * (1 - norm.cdf(|z|))` underflows to EXACTLY 0 once |z| exceeds about
    8.3, so `-log10(p)` becomes infinite for the strongest regions -- which
    then makes every aggregate statistic in Sections 11-12 infinite, and
    `perm_pvalue` returns NaN for the whole test. The log-space route stays
    finite and accurate out past |z| = 30.

    Validated against smf.ols + the Section 4 contrast machinery in 10c.
    """
    R = X.shape[0]
    n   = np.zeros((R, 8))
    mu  = np.full((R, 8), np.nan)
    ssr = np.zeros((R, 8))

    for c in range(8):
        cols = np.flatnonzero(cell_idx == c)
        if cols.size == 0:
            continue
        Xc = X[:, cols]
        n[:, c] = (~np.isnan(Xc)).sum(axis=1)
        with np.errstate(invalid="ignore"):
            mu[:, c] = np.nanmean(Xc, axis=1)
        ssr[:, c] = np.nansum((Xc - mu[:, c][:, None]) ** 2, axis=1)

    dof = n.sum(axis=1) - 8
    with np.errstate(invalid="ignore", divide="ignore"):
        pooled_var = ssr.sum(axis=1) / dof

    used = np.flatnonzero(w != 0)
    # Plain (not nan-) sum: an unoccupied cell this contrast actually touches
    # SHOULD poison the estimate to NaN rather than be silently treated as zero.
    est = (mu[:, used] * w[used]).sum(axis=1)

    n_used = n[:, used].astype(float)
    n_used[n_used == 0] = np.nan
    with np.errstate(invalid="ignore", divide="ignore"):
        var_factor = ((w[used] ** 2) / n_used).sum(axis=1)
        se = np.sqrt(pooled_var * var_factor)
        z  = est / se
    absz = np.abs(z)
    p = 2 * norm.sf(absz)
    with np.errstate(invalid="ignore"):
        neglog10p = -(np.log(2) + norm.logsf(absz)) / np.log(10)

    bad = ((np.nanmin(n[:, used], axis=1) < min_cell_n)
           | (dof <= 0) | ~np.isfinite(se) | (se == 0) | ~np.isfinite(z))
    nan = lambda v: np.where(bad, np.nan, v)
    return est, se, nan(z), nan(p), nan(neglog10p)

### 10c. Exact label permutation (simple effects and main effects)

Two knobs: which factor gets relabelled, and which factors are held fixed as
strata. Restricted permutation preserves the marginal counts of every other
factor exactly.

### CHOICE -- which factor to shuffle
The default picks the first factor that actually **varies** across the cells the
contrast touches, in the order light > genotype > sex. Light is preferred
because it is the only **experimentally assigned** factor here; sex and genotype
are subject attributes that were never randomized, so permuting them tests an
exchangeability null that could not have been realized by any experiment. That
null is still mathematically valid under H0, but it is a weaker and different
claim. Every contrast reported here varies light or genotype, so light is
shuffled wherever it varies and genotype elsewhere (e.g. `control_vs_MKO | dark | M`,
where light is held fixed). Which factor was shuffled belongs in the methods for
each number, rather than presenting the two as equivalent.

In [ ]:
def contrast_restriction(w):
    """Factor levels shared by every cell this contrast touches.

    e.g. `light_vs_dark | control | M` only touches cre/+ M cells, so this
    returns {'genotype': 'cre/+', 'sex': 'M'} -- the shuffle happens among
    control males only. Returns {} for a fully-crossed contrast."""
    touched = [CELLS[i] for i in np.flatnonzero(w != 0)]
    out = {}
    for pos, factor in enumerate(FACTORS):
        levels = {c[pos] for c in touched}
        if len(levels) == 1:
            out[factor] = levels.pop()
    return out


def default_shuffle_factor(w):
    """First factor that varies across the contrast's cells, preferring light."""
    restriction = contrast_restriction(w)
    for f in ("light", "genotype", "sex"):
        if f not in restriction:
            return f
    raise ValueError("Contrast touches only one cell -- nothing to shuffle.")


def make_permuter(contrast_name, shuffle_factor=None, strata="auto", seed=0):
    """Returns (permute_fn, info). Each permute_fn() call gives a fresh
    (n_mice,) cell-index array.

    strata="auto" -> the two factors other than shuffle_factor. Mice outside the
    contrast's restriction keep their real labels: they contribute only to the
    pooled variance and residual df, exactly as in the real fit.
    """
    ctype, w = ALL_CONTRASTS[contrast_name]
    restriction = contrast_restriction(w)
    if shuffle_factor is None:
        shuffle_factor = default_shuffle_factor(w)
    if shuffle_factor in restriction:
        raise ValueError(
            f"'{contrast_name}' holds {shuffle_factor}='{restriction[shuffle_factor]}' fixed "
            f"in every cell it touches -- shuffling it would relabel mice into cells this "
            f"contrast cannot see, producing a null unrelated to the statistic. "
            f"Try shuffle_factor='{default_shuffle_factor(w)}'."
        )
    if strata == "auto":
        strata = [f for f in FACTORS if f != shuffle_factor]

    rng = np.random.default_rng(seed)

    eligible = np.ones(len(mouse_design), dtype=bool)
    for factor, level in restriction.items():
        eligible &= (mouse_design[factor].values == level)

    strat_keys = (list(zip(*[mouse_design[f].values for f in strata]))
                  if strata else [()] * len(mouse_design))
    groups = {}
    for i, k in enumerate(strat_keys):
        if eligible[i]:
            groups.setdefault(k, []).append(i)
    group_idx = [np.array(v) for v in groups.values()]

    base_labels = mouse_design[shuffle_factor].values.copy()
    factor_pos = FACTORS.index(shuffle_factor)
    base_triples = [list(t) for t in zip(mouse_design["genotype"].values,
                                         mouse_design["sex"].values,
                                         mouse_design["light"].values)]

    # Resolution ceiling: product over strata of C(n_g, n_g_levelA)
    n_distinct = 1
    for idx in group_idx:
        vals = base_labels[idx]
        lv = pd.unique(vals)
        if len(lv) == 2:
            n_distinct *= comb(len(idx), int((vals == lv[0]).sum()))

    def permute_fn():
        labels = base_labels.copy()
        for idx in group_idx:
            labels[idx] = rng.permutation(labels[idx])
        out = np.empty(len(base_triples), dtype=int)
        for i, t in enumerate(base_triples):
            t = t.copy()
            t[factor_pos] = labels[i]
            out[i] = CELL_INDEX[tuple(t)]
        return out

    info = {"shuffle_factor": shuffle_factor, "strata": strata, "restriction": restriction,
            "n_shuffled_mice": int(eligible.sum()), "n_strata": len(group_idx),
            "max_distinct_permutations": n_distinct}
    return permute_fn, info


# ---- Validation: the fast path must reproduce the Section 5 pipeline exactly ----
def validate_fast_path(contrast_name, tol=1e-6, verbose=True):
    _, w = ALL_CONTRASTS[contrast_name]
    p_fast = contrast_fast(XMAT, OBS_CELL_IDX, w)[3]
    fast = pd.Series(p_fast, index=PERM_REGIONS).dropna()
    ref = all_results[all_results["contrast"] == contrast_name].set_index("Acronym")["p"]
    shared = fast.index.intersection(ref.index)
    max_diff = (fast.loc[shared] - ref.loc[shared]).abs().max()
    if verbose:
        print(f"  {contrast_name:<38} {len(shared):>4} regions   "
              f"max |p_fast - p_pipeline| = {max_diff:.2e}")
    assert max_diff < tol, (
        f"Fast closed form disagrees with the Section 5 OLS pipeline for "
        f"'{contrast_name}' (max diff {max_diff:.2e}). STOP -- nothing downstream "
        f"is trustworthy until this matches."
    )
    return max_diff


print("Validating the fast path against Section 5, for every reported contrast:")
for _c in ALL_CONTRASTS:
    validate_fast_path(_c)
print("\nAll contrasts validated.")

### 10d. Freedman-Lane, for interaction terms

**The problem.** You want to test one term -- say the three-way -- in a model
that also contains main effects and two-ways that are genuinely non-null. No
shuffle of the raw labels leaves those intact while nulling only the target:
relabelling light scrambles the light main effect too. So a plain label shuffle
tests a *composite* null ("light has no effect anywhere"), and a significant
result licenses only "the three-way statistic is more extreme than under a
no-light-effect null" -- not "the three-way term is non-null", which is the
sentence a reviewer will assume you meant.

**The fix (Freedman & Lane 1983; Winkler et al. 2014, NeuroImage).** Permute
*residuals from the reduced model* instead of labels:

1. Fit the **reduced** model -- every term except the one being tested. Keep
   fitted values `Y_r` and residuals `e_r`.
2. Permute `e_r` across mice -> `e_r*`.
3. Build `Y* = Y_r + e_r*`. Every real main effect and two-way is still sitting
   there unshuffled; the only part that could have carried the target term --
   the leftover -- has been randomly reassigned.
4. Refit the **full** model to `Y*` and extract the target contrast.
5. Repeat, giving a null for the target *conditional on the nuisance structure
   being real*.

The intuition: reduced-model residuals are what is left once everything except
the effect of interest is accounted for, and under H0 those are exchangeable.

### CHOICE -- general contrast partitioning
Rather than special-casing each term, the reduced model is derived from the
contrast vector itself. The 8 cells are re-expressed in an orthogonal +-1 basis
(intercept, G, S, L, GS, GL, SL, GSL); a contrast `w` over cell means becomes
`c = B'w` in that coefficient space; the nuisance design is `Z @ null_space(c')`,
i.e. everything orthogonal to the direction being tested. For the three-way this
reduces to exactly "drop the GSL column". For a **split** two-way like
`geno:sex | light` -- which is *not* a single basis column but the combination
`(GS + GSL)/2` -- it does the right thing automatically, which hand-coding would
not.

### Calibration check -- measured, not asserted

Simulated at this design (n=5/cell, 60 regions) with a real light main effect, a
real `sex:light`, and a real `geno:light` present, but the three-way set to
**exactly zero**. A correct test should return ~uniform p. 60 reps:

| method | statistic | frac p < .05 | median p | |
|---|---|---|---|---|
| plain label shuffle | `mean_z` | 0.00 | **0.92** | badly conservative |
| plain label shuffle | `sum` | 0.00 | 0.33 | conservative |
| **Freedman-Lane** | `mean_z` | **0.05** | **0.53** | calibrated |
| **Freedman-Lane** | `sum` | 0.03 | 0.47 | calibrated |

The failure mode is the opposite of what you might expect: the label shuffle is
not anticonservative here, it is nearly **powerless**. Because its null contains
all the real light effects, the null distribution of the three-way statistic is
enormously inflated relative to the truth, and the observed value sits at the
40th percentile of it. Median p = 0.92 means that scheme would have missed a real
three-way almost regardless of size.

Power at a true three-way of 0.45, Freedman-Lane, `mean_z`: **1.00**, median
p = 0.002.

So Freedman-Lane is not a technicality here -- it is the difference between
having a usable test of the three-way and not having one.

### FLAG -- approximate, not exact
Reduced-model residuals are not perfectly exchangeable: they are constrained
orthogonal to the reduced design, which bites at small n. Freedman-Lane is the
best-behaved of the competing schemes in Winkler et al.'s comparison, not an
exact test. Say so in the methods rather than describing it as exact.

In [ ]:
# Orthogonal +-1 basis over the 8 cells: intercept, G, S, L, GS, GL, SL, GSL
BASIS_NAMES = ["intercept", "G", "S", "L", "GS", "GL", "SL", "GSL"]
B8 = np.array([[1,
                code_map[g], code_map[s], code_map[l],
                code_map[g] * code_map[s], code_map[g] * code_map[l], code_map[s] * code_map[l],
                code_map[g] * code_map[s] * code_map[l]]
               for g, s, l in CELLS], dtype=float)
assert np.allclose(B8.T @ B8, 8 * np.eye(8)), "Cell basis is not orthogonal."


def freedman_lane_setup(w, cell_idx=None):
    """Reduced-model fitted values and residuals for the contrast `w`.

    Returns (Yhat_r, E_r, nuisance_terms) where Yhat_r and E_r are both
    (n_regions, n_mice). `nuisance_terms` lists which basis directions the
    reduced model retains, printed so you can eyeball that the right term was
    removed.
    """
    if cell_idx is None:
        cell_idx = OBS_CELL_IDX
    assert REQUIRE_COMPLETE, (
        "Freedman-Lane shares one reduced-model hat matrix across regions, which "
        "is only valid when every region has the same design. Set "
        "REQUIRE_COMPLETE=True, or fall back to method='label'."
    )
    c = B8.T @ w                       # contrast in coefficient space
    Cu = null_space(c[None, :])        # (8, 7) nuisance directions
    Z = B8[cell_idx]                   # (n_mice, 8) mouse-level design
    W = Z @ Cu                         # (n_mice, 7) reduced design
    H = W @ np.linalg.pinv(W)          # hat matrix, shared by all regions

    Yhat_r = XMAT @ H.T
    E_r = XMAT - Yhat_r

    # Which basis directions the removed direction loads on, for reporting
    load = np.abs(c) / (np.abs(c).max() + 1e-12)
    removed = [nm for nm, v in zip(BASIS_NAMES, load) if v > 1e-8]
    return Yhat_r, E_r, removed


def make_fl_permuter(w, seed=0, strata=None):
    """Returns (permute_fn, info). permute_fn() gives a fresh Y* matrix.

    ### CHOICE ###
    Residuals are permuted FREELY across all mice (Winkler et al.'s default),
    not within strata. Free permutation is what makes the reduced model's
    structure the only thing preserved; restricting the shuffle would preserve
    additional structure the null is not supposed to keep. `strata` is exposed
    for sensitivity checks but should normally stay None.
    """
    Yhat_r, E_r, removed = freedman_lane_setup(w)
    rng = np.random.default_rng(seed)
    n_mice = XMAT.shape[1]

    if strata is None:
        groups = [np.arange(n_mice)]
    else:
        keys = list(zip(*[mouse_design[f].values for f in strata]))
        gmap = {}
        for i, k in enumerate(keys):
            gmap.setdefault(k, []).append(i)
        groups = [np.array(v) for v in gmap.values()]

    def permute_fn():
        perm = np.arange(n_mice)
        for idx in groups:
            perm[idx] = rng.permutation(idx)
        return Yhat_r + E_r[:, perm]

    info = {"method": "freedman_lane", "removed_directions": removed,
            "reduced_model_rank": 7, "residual_strata": strata,
            "max_distinct_permutations": np.inf}
    return permute_fn, info


# ---- Show what gets removed for each interaction term ----
print("Freedman-Lane reduced models (direction removed from the 8-cell basis):")
for cname in INTERACTION_CONTRASTS:
    _, w = ALL_CONTRASTS[cname]
    _, _, removed = freedman_lane_setup(w)
    print(f"  {cname:<26} removes: {'+'.join(removed)}")

### 10e. `run_permutation()`

In [ ]:
@dataclass
class PermResult:
    contrast: str
    method: str
    regions: np.ndarray            # (n_regions,)
    obs_neglog10p: np.ndarray      # (n_regions,)
    obs_z: np.ndarray              # (n_regions,)
    obs_est: np.ndarray            # (n_regions,)
    null_neglog10p: np.ndarray     # (n_perm, n_regions)
    null_z: np.ndarray             # (n_perm, n_regions)
    info: dict = field(default_factory=dict)

    @property
    def n_perm(self):
        return self.null_neglog10p.shape[0]

    def region_table(self):
        return pd.DataFrame({"Acronym": self.regions, "estimate": self.obs_est,
                             "z": self.obs_z, "neg_log10_p": self.obs_neglog10p})


def run_permutation(contrast_name, n_perm=2000, method="auto",
                    shuffle_factor=None, strata="auto", seed=0, verbose=True):
    """Permutation null for one contrast across all PERM_REGIONS.

    method="auto" -> Freedman-Lane for interaction terms (where a plain label
    shuffle would test a composite null), exact label permutation for simple
    and main effects (where it is the right null already).
    """
    ctype, w = ALL_CONTRASTS[contrast_name]
    if method == "auto":
        method = "freedman_lane" if ctype == "interaction" else "label"

    if method == "label":
        permute_fn, info = make_permuter(contrast_name, shuffle_factor, strata, seed)
        gen = lambda: contrast_fast(XMAT, permute_fn(), w)
    elif method == "freedman_lane":
        permute_fn, info = make_fl_permuter(w, seed=seed,
                                            strata=None if strata == "auto" else strata)
        gen = lambda: contrast_fast(permute_fn(), OBS_CELL_IDX, w)
    else:
        raise ValueError(method)
    info["method"] = method

    obs_est, _, obs_z, obs_p, obs_nlp = contrast_fast(XMAT, OBS_CELL_IDX, w)
    n_gated = int(np.isnan(obs_p).sum())

    if verbose:
        print(f"Contrast:   {contrast_name}   [{ctype}]")
        print(f"Method:     {method}")
        if method == "label":
            print(f"  shuffling {info['shuffle_factor']} within {info['strata']} "
                  f"({info['n_shuffled_mice']} mice, {info['n_strata']} strata)")
            if info["restriction"]:
                print(f"  restricted to {info['restriction']}")
            ceil_ = info["max_distinct_permutations"]
            print(f"  {ceil_} distinct labellings -> smallest attainable p ~ {1/(ceil_+1):.3g}")
            if ceil_ < n_perm:
                print(f"  NOTE: fewer distinct labellings than n_perm={n_perm}. The loop is "
                      f"resampling the same {ceil_} repeatedly; p CANNOT go below the ceiling "
                      f"above no matter how many permutations you run. Report the ceiling.")
        else:
            print(f"  reduced model removes: {'+'.join(info['removed_directions'])} "
                  f"(rank {info['reduced_model_rank']} nuisance design)")
        print(f"Regions:    {len(PERM_REGIONS)} ({n_gated} gated to NaN by MIN_CELL_N)")

    null_nlp = np.empty((n_perm, len(PERM_REGIONS)))
    null_z   = np.empty((n_perm, len(PERM_REGIONS)))
    for i in range(n_perm):
        _, _, z_i, _, nlp_i = gen()
        null_nlp[i] = nlp_i
        null_z[i]   = z_i
        if verbose and (i + 1) % max(1, n_perm // 4) == 0:
            print(f"  {i+1}/{n_perm} permutations")

    return PermResult(contrast=contrast_name, method=method, regions=PERM_REGIONS,
                      obs_neglog10p=obs_nlp, obs_z=obs_z, obs_est=obs_est,
                      null_neglog10p=null_nlp, null_z=null_z,
                      info={**info, "contrast_type": ctype, "n_perm": n_perm,
                            "seed": seed, "n_gated": n_gated})


def perm_pvalue(null_vals, obs_val, tail="greater"):
    """Phipson & Smyth (2010): (1 + #{null as extreme}) / (n + 1).

    ### CHOICE ###
    NOT `(null >= obs).mean()`, which returns exactly 0 when no permutation
    beats the observed. A p-value of 0 is not a valid claim from a finite
    permutation set; the +1 correction reports the resolution ceiling honestly.
    ### FIXED FROM PREVIOUS VERSION ###
    """
    v = np.asarray(null_vals, dtype=float)
    v = v[np.isfinite(v)]
    n = len(v)
    if n == 0 or not np.isfinite(obs_val):
        return np.nan
    if tail == "greater":
        k = (v >= obs_val).sum()
    elif tail == "less":
        k = (v <= obs_val).sum()
    elif tail == "two-sided":
        centre = np.median(v)
        k = (np.abs(v - centre) >= abs(obs_val - centre)).sum()
    else:
        raise ValueError(tail)
    return (1 + k) / (n + 1)

## 11. Global breadth test

Is the effect unusually **broad** across regions, relative to the null?

### CHOICE -- signed `mean(z)` is the primary statistic, not `sum(-log10 p)`

`-log10 p` is **direction-blind**. A region set where half the regions go up and
half go down scores identically to one where they all move together. For a
crossover claim -- "control males activate brainwide, control females do not,
and MKO reverses it" -- that is precisely the distinction you need, and the
unsigned statistic cannot see it.

So the primary panel is `mean(z)` across regions, two-sided: a large positive
value means a *consistent brainwide shift in one direction*. `sum(-log10 p)` is
kept as a secondary panel because it has more power against a broad effect of
*mixed* sign, and the two disagreeing is itself informative -- a big
`sum(-log10 p)` with `mean(z)` near zero says "lots of regions are doing
something, but not the same something", which is a spatial-redistribution
finding, not a gain finding. That is the same scalar-gain versus
spatial-pattern distinction the PLSC pipeline makes.

### FLAG
The global breadth test is sensitive to per-mouse whole-brain gain (Section 7c).
If one animal stains uniformly hot, every region shifts together and `mean(z)`
picks it up as "broad effect". Read Section 7c and Section 11 together, and if
the gain check looks bad, re-run this on per-mouse-centred data before believing
the breadth result.

In [ ]:
def perm_statistic(nlp, z=None, mask=None, stat="mean_z"):
    """Collapse a (..., n_regions) array to a scalar (or per-permutation vector).

      "mean_z"     mean signed z            <- DEFAULT: directional, for crossover
      "mean_absz"  mean |z|                    magnitude regardless of direction
      "sum"        sum(-log10 p)               direction-blind breadth
      "mean"       mean(-log10 p)
      "count_sig"  #{p < 0.05}
      "sum_p"      sum(p)                      direction inverted: small = evidence
    """
    if mask is not None:
        nlp = nlp[..., mask]
        z = None if z is None else z[..., mask]
    if stat == "mean_z":     return np.nanmean(z, axis=-1)
    if stat == "mean_absz":  return np.nanmean(np.abs(z), axis=-1)
    if stat == "sum":        return np.nansum(nlp, axis=-1)
    if stat == "mean":       return np.nanmean(nlp, axis=-1)
    if stat == "count_sig":  return np.nansum(nlp > -np.log10(0.05), axis=-1)
    if stat == "sum_p":      return np.nansum(10.0 ** (-nlp), axis=-1)
    raise ValueError(stat)


STAT_LABEL = {"mean_z": "mean signed z across regions",
              "mean_absz": "mean |z| across regions",
              "sum": "sum(-log10 p) across regions",
              "mean": "mean(-log10 p) across regions",
              "count_sig": "count(p < 0.05) across regions",
              "sum_p": "sum(p) across regions"}


# Statistics that carry a SIGN. A consistent shift in either direction is the
# finding, so these are two-sided; unsigned magnitude statistics are one-sided
# greater. Kept as a set so adding a statistic in one place cannot leave its tail
# behind in another.
SIGNED_STATS = {"mean_z", "mean_z_diff", "mean_z_vs_all"}


def default_tail(stat):
    if stat in SIGNED_STATS:
        return "two-sided"
    if stat == "sum_p":
        return "less"
    return "greater"


def plot_global_null(res, stat="mean_z", tail=None, ax=None, bins=45, color="#4C72B0"):
    tail = tail or default_tail(stat)
    obs  = perm_statistic(res.obs_neglog10p, res.obs_z, stat=stat)
    null = perm_statistic(res.null_neglog10p, res.null_z, stat=stat)
    pval = perm_pvalue(null, obs, tail=tail)

    created = ax is None
    if created:
        _, ax = plt.subplots(figsize=(6, 4.2))
    ax.hist(null, bins=bins, color=color, alpha=0.85, edgecolor="white")
    ax.axvline(obs, color="#C44E52", lw=2.2, label=f"observed = {obs:.3f}")
    ax.axvline(np.nanmean(null), color="gray", lw=1, ls=":", label="null mean")
    ax.set(xlabel=STAT_LABEL[stat], ylabel="permutations",
           title=f"{res.contrast}\n{res.method} | perm p = {pval:.4f} ({res.n_perm} perms)")
    ax.legend(fontsize=8)
    if created:
        plt.tight_layout()

    ceil_ = res.info.get("max_distinct_permutations", np.inf)
    return {"contrast": res.contrast, "method": res.method, "stat": stat,
            "observed": obs, "null_mean": np.nanmean(null), "null_sd": np.nanstd(null),
            "z_vs_null": (obs - np.nanmean(null)) / np.nanstd(null),
            "perm_p": pval, "tail": tail, "n_perm": res.n_perm,
            "resolution_floor": 1 / (ceil_ + 1)}

In [ ]:
# ### CHOICE ###
# n_perm=2000 is a reasonable interactive default. For label-permutation
# contrasts, watch the resolution ceiling printed below: with few mice per
# stratum the number of DISTINCT labellings can be smaller than n_perm, and no
# amount of extra permutations pushes p below 1/(n_distinct+1). Freedman-Lane
# has no such ceiling (residuals permute n_mice! ways) but is approximate.
N_PERM = 2000

PERM_CONTRASTS = [
    "light_vs_dark | control | M",
    "light_vs_dark | control | F",
    "light_vs_dark | MKO | M",
    "light_vs_dark | MKO | F",
    "sex:light | control",
    "sex:light | MKO",
    "geno:sex:light", "light (main effect)", "geno:sex | light",
    "control_vs_MKO | light | F", "control_vs_MKO | light | M",
    "geno:sex | dark", "control_vs_MKO | dark | F", "control_vs_MKO | dark | M",
]

assert all(c in ALL_CONTRASTS for c in PERM_CONTRASTS), \
    "a PERM_CONTRASTS entry is not in the reported contrast set"


perm_results = {}
for cname in PERM_CONTRASTS:
    print("=" * 70)
    perm_results[cname] = run_permutation(cname, n_perm=N_PERM, seed=0)
    print()

In [ ]:
def plot_global_grid(results, stat, fname, ncols=4):
    n = len(results)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.2 * nrows), squeeze=False)
    axes_flat = axes.reshape(-1)
    rows = [plot_global_null(res, stat=stat, ax=ax)
            for ax, res in zip(axes_flat, results.values())]
    for ax in axes_flat[n:]:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT_DIR / fname, dpi=200)
    plt.show()
    return rows


print("PRIMARY: signed mean(z) -- directional, two-sided")
rows_primary = plot_global_grid(perm_results, "mean_z", "permutation_null_mean_z.png")

print("\nSECONDARY: sum(-log10 p) -- direction-blind breadth")
rows_secondary = plot_global_grid(perm_results, "sum", "permutation_null_sum.png")

global_perm_table = pd.DataFrame(rows_primary + rows_secondary)
print()
print(global_perm_table.to_string(index=False))
global_perm_table.to_csv(OUT_DIR / "global_permutation_results.csv", index=False)

# ### READ THESE TOGETHER ###
# A contrast with a large `sum` but a `mean_z` near zero has many regions
# responding in INCONSISTENT directions -- spatial redistribution, not a
# brainwide gain change. A contrast significant on both is a consistent
# brainwide shift. Reporting only `sum` cannot distinguish them.

## 12. Enrichment testing on pre-specified region sets

### FLAG -- why `sum(-log10 p)` inside a subset is the wrong enrichment statistic

Take the obvious statistic: `sum(-log10 p)` over regions in S, compared to a
label-permutation null. Under H0 ("no effect anywhere") that is a valid test.
But it does **not** test enrichment. If light genuinely affects the brain
broadly -- which is the whole finding -- then *every* subset of comparable size
looks significant against that null, because the null assumes no effect at all.
It would return "significant" for MA/NDB/SI and equally for three random
cerebellar leaves. Same failure mode as the naive Fisher's exact test this is
meant to replace, in a new costume.

Enrichment is a claim about **differential** concentration: S carries more of
the effect than the rest of the brain. So the statistic must be a contrast, not
a level:

```
T = mean(z | in S)  -  mean(z | not in S)
```

invariant to brainwide gain, since a uniform effect lifts both terms equally.
That is `stat="mean_z_diff"`, the default. Signed rather than `-log10 p` for the
same reason as Section 11: a set whose regions move *consistently* in one
direction is a different and stronger finding than one whose regions merely move.

### The two nulls

| | randomizes | holds fixed | question |
|---|---|---|---|
| **Label null** (12c) | design labels across mice | region identity, anatomy, inter-regional correlation | is the S-vs-background gap larger than relabelling produces? |
| **Region null** (12e) | which regions are in S | observed per-region statistics | is S more concentrated than a size-matched region set? |

### Simulation check -- this is not just an argument

Simulating this design directly (n=5/cell, 60 regions, 3-region subset, noise
matched to a plausible log-density SD), over 15-40 reps per scenario -- read as
indicative magnitudes, not precise rates:

| scenario | statistic + null | frac p < .05 | median p |
|---|---|---|---|
| pure noise | global breadth, label null | **0.05** | 0.46 |
| uniform brainwide effect, **zero** true enrichment | `sum_in`, label null | **0.53** | 0.03 |
| uniform brainwide effect, **zero** true enrichment | `mean_diff`, label null | 0.20 | 0.21 |
| uniform brainwide effect, **zero** true enrichment | `mean_diff`, region null | 0.10 | **0.52** |
| genuine enrichment | `mean_diff`, label null | 1.00 | 0.01 |
| genuine enrichment | `mean_diff`, region null | 1.00 | 0.00 |

Three things follow:

1. The global breadth test is correctly calibrated (0.05 at alpha = 0.05 under
   pure noise). Section 11 is fine as-is.
2. `sum_in` under a label null declares enrichment **over half the time when
   there is none**, median p = .03. Do not report it as enrichment.
3. `mean_diff` under the **label** null is still somewhat anticonservative
   (median p .21, not .50) when a strong global effect exists, because the label
   null contains no real between-region effect heterogeneity and so
   underestimates the variance of a subset-vs-background difference. The
   **region** null, which conditions on the observed map, is correctly centred
   (median .52).

So: **region null leads the enrichment claim; label null is reported alongside**
as the complement that respects inter-regional correlation. Do not let the label
null carry an enrichment claim on its own.

---

### CHOICE -- subset vs WHOLE BRAIN, not subset vs complement

The default statistic is now

```
T = mean(z | in S)  -  mean(z | all regions)          # "mean_z_vs_all"
```

rather than the complement form

```
T = mean(z | in S)  -  mean(z | not in S)             # "mean_z_diff"
```

Both are contrasts rather than levels, so both are invariant to a uniform
brainwide gain, which is the property that matters. They differ only by a
constant:

```
mean_z_vs_all  =  (1 - k/N) * mean_z_diff
```

with `k` the set size and `N` the number of regions. Because both nulls hold `k`
and `N` fixed -- the region null draws size-matched sets, and restricted label
shuffles preserve cell counts exactly so gating is permutation-invariant -- that
constant multiplies the observed statistic and every null draw identically.
**The permutation p-value and `z_vs_null` are unchanged.** 12b asserts this.

The reason to prefer it is reporting, not inference: it reads directly as "this
set sits X above the brain-wide average", and it stops inflating the effect size
for large sets. For a subdivision covering 40% of regions the complement form is
1.67x larger for purely arithmetic reasons, which makes cross-set comparison
misleading. Both columns are carried through to the output table.

Note this is still a **competitive** test in the Goeman & Buhlmann sense -- S is
compared against the rest of the map, not against a fixed null of no effect. Only
the reference point moved.

### 12a. Define and validate the region sets

### CHOICE -- these must be pre-specified
"Pre-determined" is the entire basis for treating these as confirmatory. Fix
them from anatomy and prior literature **before** looking at results, and write
them down somewhere citable. Any set assembled by reading off a volcano plot is
exploratory and must be labelled as such -- the same PLSC-circularity discipline
applies here.

The validation cell below checks every acronym against the analysis data and
against the atlas, and suggests near-misses, so a set does not silently shrink
from 11 regions to 4 because of a naming mismatch.

In [ ]:
REGION_SETS = {
    "Neuromodulatory source nuclei": ["NDB", "SI", "MA","MS", "SO", "PVH", "DR", "CLI",
                                      "LC", "SNc", "VTA", "PPN", "LDT"],

    "ACh nuclei": ["NDB", "SI", "MA","MS", "PPN", "LDT"],

    "NE/SE nuclei": [ "DR", "CLI","LC"],

    "DA nuclei": [ "SNc", "VTA"],

    "Oxt nuclei": ["SO", "PVH"],

    "retinorecipient_full": ["AHN", "BST", "DT", "IGL", "LGd-sh", "LGv",
                                      "LH", "LP", "LT", "MEA", "NOT", "OP", "PAG", "SBPV", "SCH", "SCop", "SGN", "SI", "SO", "VLPO", "ZI",
                            "AAA", "AD", "APN", "CL", "ICd", "DR", "LHA", "LHN", "VPL", "MPT", "MRN", "MT", "PB", "PN", "PP", "PPT",
                            "RCH", "SubG", "MA", "NDB"],

    "ipRGC_targets_full": ["AHN", "BST", "DT", "IGL", "LGd-sh", "LGv",
                                      "LH", "LP", "LT", "MEA", "NOT", "OP", "PAG", "SBPV", "SCH", "SCop", "SGN", "SI", "SO", "VLPO", "ZI"],

    "image_forming_full": ["VISal", "VISam", "VISl", "VISp", "VISpl", "VISpm", "VISli", "VISpor", "VISa", "VISrl", "TEa",
                                      "RSPagl", "RSPd", "RSPv", "PTLp"],
}


def validate_region_sets(region_sets, verbose=True):
    """Check every acronym against (a) the permutation region set, (b) all
    acronyms present in the data, (c) the atlas -- and suggest near-misses."""
    data_acronyms = set(d["Acronym"].unique())
    report = []
    for name, acrs in region_sets.items():
        for a in acrs:
            if a in REGION_POS:
                status = "ok"
            elif a in data_acronyms:
                status = "in data, gated out (missingness / MIN_CELL_N)"
            else:
                try:
                    bg_atlas.structures[a]
                    status = "valid atlas acronym, ABSENT from data"
                except KeyError:
                    status = "NOT a valid atlas acronym"
            suggestion = ""
            if status != "ok":
                near = difflib.get_close_matches(a, sorted(data_acronyms), n=3, cutoff=0.6)
                if near:
                    suggestion = "did you mean: " + ", ".join(near)
            report.append({"set": name, "acronym": a, "status": status,
                           "suggestion": suggestion})
    report = pd.DataFrame(report)

    if verbose:
        bad = report[report["status"] != "ok"]
        n_ok = (report["status"] == "ok").sum()
        print(f"{n_ok} / {len(report)} acronyms resolved cleanly.")
        if len(bad):
            print("\nUnresolved:")
            print(bad.to_string(index=False))
            print("\n!! Fix these before running anything below -- a set that quietly "
                  "shrinks changes the test without changing anything you'd notice "
                  "in the output.")
        print("\nEffective set sizes:")
        for name, acrs in region_sets.items():
            n = sum(a in REGION_POS for a in acrs)
            print(f"  {name:<32} {n} / {len(acrs)}")
    return report


region_set_report = validate_region_sets(REGION_SETS)

### 12b. Subset statistics

In [ ]:
def subset_statistic(nlp, z, in_mask, stat="mean_z_vs_all"):
    """Statistic for a region subset, broadcasting over a leading perm axis.

    Subset vs WHOLE BRAIN (the default family):
      "mean_z_vs_all"     mean(z | in S)      - mean(z | all)      <- DEFAULT
      "mean_absz_vs_all"  mean(|z| | in S)    - mean(|z| | all)
      "mean_vs_all"       mean(-log10 p | S)  - mean(-log10 p | all)

    Subset vs COMPLEMENT (kept: identical p, rescaled effect size):
      "mean_z_diff"       mean(z | in S)      - mean(z | out)
      "mean_absz_diff"    mean(|z| | in S)    - mean(|z| | out)
      "mean_diff"         mean(-log10 p | S)  - mean(-log10 p | out)

    Levels, NOT contrasts -- confounded with brainwide gain, never an enrichment
    claim (see the Section 12 header):
      "mean_in"           mean(-log10 p | in S)
      "sum_in"            sum(-log10 p | in S)

    The `_vs_all` and `_diff` forms of the same quantity differ by the positive
    constant (1 - k/N), so any permutation p computed from them is identical.
    """
    out_mask = ~in_mask
    az = None if z is None else np.abs(z)

    if stat == "sum_in":   return np.nansum(nlp[..., in_mask], axis=-1)
    if stat == "mean_in":  return np.nanmean(nlp[..., in_mask], axis=-1)

    if stat == "mean_diff":
        return np.nanmean(nlp[..., in_mask], -1) - np.nanmean(nlp[..., out_mask], -1)
    if stat == "mean_z_diff":
        return np.nanmean(z[..., in_mask], -1) - np.nanmean(z[..., out_mask], -1)
    if stat == "mean_absz_diff":
        return np.nanmean(az[..., in_mask], -1) - np.nanmean(az[..., out_mask], -1)

    if stat == "mean_vs_all":
        return np.nanmean(nlp[..., in_mask], -1) - np.nanmean(nlp, -1)
    if stat == "mean_z_vs_all":
        return np.nanmean(z[..., in_mask], -1) - np.nanmean(z, -1)
    if stat == "mean_absz_vs_all":
        return np.nanmean(az[..., in_mask], -1) - np.nanmean(az, -1)

    raise ValueError(stat)


# Pairing used by the summary tables, so both effect sizes are always reported.
VS_ALL_PARTNER = {"mean_z_vs_all": "mean_z_diff",
                  "mean_absz_vs_all": "mean_absz_diff",
                  "mean_vs_all": "mean_diff"}


def resolve_regions(acronyms, res, label="region set", warn=True):
    """Map a requested acronym list onto the permutation region index."""
    present = [a for a in acronyms if a in REGION_POS]
    missing = [a for a in acronyms if a not in REGION_POS]
    if warn and missing:
        print(f"  ! {label}: {len(missing)} acronym(s) unresolved: {missing}")
    mask = np.zeros(len(res.regions), dtype=bool)
    mask[[REGION_POS[a] for a in present]] = True
    return mask, present, missing


# ---- The (1 - k/N) identity, asserted rather than argued -------------------
# If this ever fails, the two statistic families have stopped being related by a
# constant and the "p is unchanged" claim in the Section 12 header is void.
def _assert_vs_all_identity(res, n_test=3, seed=0):
    rng_ = np.random.default_rng(seed)
    N = len(res.regions)
    for k in (5, 25, max(60, N // 4)):
        if k >= N:
            continue
        m = np.zeros(N, dtype=bool)
        m[rng_.choice(N, k, replace=False)] = True
        f = 1 - k / N
        for a, b in VS_ALL_PARTNER.items():
            o1 = subset_statistic(res.obs_neglog10p, res.obs_z, m, a)
            o2 = subset_statistic(res.obs_neglog10p, res.obs_z, m, b)
            assert np.allclose(o1, f * o2, rtol=1e-9, atol=1e-12), \
                f"{a} != (1-k/N) * {b} at k={k}"
            n1 = subset_statistic(res.null_neglog10p, res.null_z, m, a)
            n2 = subset_statistic(res.null_neglog10p, res.null_z, m, b)
            assert np.allclose(n1, f * n2, rtol=1e-9, atol=1e-12), \
                f"null {a} != (1-k/N) * {b} at k={k}"
            p1 = perm_pvalue(n1, o1, tail=default_tail(a))
            p2 = perm_pvalue(n2, o2, tail=default_tail(b))

            # ### FIXED ###
            # This used to demand `p1 == p2` exactly, and it failed in practice
            # (e.g. 0.33233 vs 0.32384 at k=67) for a floating-point reason
            # rather than a statistical one. The two-sided branch of
            # `perm_pvalue` centres on `np.median(v)`, and for an even-length
            # null the median is the average of the two middle values -- a
            # rounding step that is NOT exactly proportional under the (1 - k/N)
            # rescaling. A restricted label permutation also draws from a small
            # number of DISTINCT labellings, so the null holds large groups of
            # exactly tied values; a one-ULP shift in the centre flips the `>=`
            # for a whole tied group at once, moving p by several draws.
            #
            # The identity itself is intact -- the two `np.allclose` assertions
            # above are the substantive check and they hold to 1e-9. The bound
            # here is the honest one: the two p-values may differ only by the
            # draws sitting exactly on the decision boundary.
            def _boundary_ties(null, obs):
                v = np.asarray(null, float)
                v = v[np.isfinite(v)]
                c = np.median(v)
                return int(np.isclose(np.abs(v - c), abs(obs - c),
                                      rtol=1e-9, atol=1e-12).sum()), len(v)

            t1, nv1 = _boundary_ties(n1, o1)
            t2, nv2 = _boundary_ties(n2, o2)
            bound = (max(t1, t2) + 1) / (min(nv1, nv2) + 1)
            assert abs(p1 - p2) <= bound, (
                f"perm p differs between {a} ({p1}) and {b} ({p2}) at k={k} by more "
                f"than the {max(t1, t2)} boundary-tied draw(s) can explain "
                f"(bound {bound:.4f}) -- the two statistic families have genuinely "
                f"stopped being related by a positive constant.")
    print("OK  mean_*_vs_all == (1 - k/N) * mean_*_diff, and permutation p agrees")
    print("    between the two families to within the draws tied at the decision")
    print("    boundary. Switching statistic changes the reported effect size only.")


_assert_vs_all_identity(perm_results[list(perm_results)[0]])

### 12c. Label null: subset vs background

In [ ]:
def enrichment_label_null(res, acronyms, stat="mean_z_diff", tail=None,
                          label=None, ax=None, plot=True, bins=45):
    label = label or f"{len(acronyms)} regions"
    tail = tail or default_tail(stat)
    in_mask, present, missing = resolve_regions(acronyms, res, label=label)
    if in_mask.sum() == 0:
        print(f"  ! {label}: no regions resolved -- skipping"); return None
    if in_mask.sum() < 3:
        print(f"  ! {label}: only {in_mask.sum()} region(s). A set this small gives the "
              f"test very little resolution -- treat the result as descriptive.")

    obs  = subset_statistic(res.obs_neglog10p, res.obs_z, in_mask, stat)
    null = subset_statistic(res.null_neglog10p, res.null_z, in_mask, stat)
    pval = perm_pvalue(null, obs, tail=tail)

    if plot:
        created = ax is None
        if created:
            _, ax = plt.subplots(figsize=(6, 4.2))
        ax.hist(null, bins=bins, color="#55A868", alpha=0.85, edgecolor="white")
        ax.axvline(obs, color="#C44E52", lw=2.2, label=f"observed = {obs:.3f}")
        ax.axvline(np.nanmean(null), color="gray", lw=1, ls=":", label="null mean")
        ax.set(xlabel=stat, ylabel="permutations",
               title=f"{label} (n={in_mask.sum()})\n{res.contrast} | LABEL null | "
                     f"p = {pval:.4f}")
        ax.legend(fontsize=8)
        if created:
            plt.tight_layout()

    return {"set": label, "contrast": res.contrast, "null_type": "label", "stat": stat,
            "n_regions": int(in_mask.sum()), "n_missing": len(missing), "observed": obs,
            "null_mean": np.nanmean(null), "null_sd": np.nanstd(null),
            "z_vs_null": (obs - np.nanmean(null)) / np.nanstd(null),
            "perm_p": pval, "tail": tail, "n_draws": res.n_perm}

### 12d. CCF centroids, for the spatial region null

One pass over the annotation volume, accumulating per-label voxel counts and
coordinate sums by `np.bincount` slice-by-slice, then aggregating each region's
descendants into a voxel-count-weighted centroid. Cached, because it takes tens
of seconds and never changes.

In [ ]:
def build_region_centroids(acronyms, cache_path=None, force=False):
    """(n_regions, 3) centroids in microns, aligned to `acronyms` order.

    Regions with no voxels in the annotation get NaN -- they cannot seed or join
    a spatial draw, and are reported rather than silently dropped."""
    cache = Path(cache_path or (OUT_DIR / "ccf_centroids.npz"))
    if cache.exists() and not force:
        z = np.load(cache, allow_pickle=True)
        if list(z["acronyms"]) == list(acronyms):
            print(f"Loaded cached centroids: {cache}")
            return z["centroids"]
        print("Cached centroid acronym list does not match -- rebuilding.")

    ann = bg_atlas.annotation
    res_um = np.array(bg_atlas.resolution, dtype=float)

    labels = np.unique(ann)
    lut = np.zeros(int(labels.max()) + 1, dtype=np.int64)
    lut[labels] = np.arange(len(labels))
    nL = len(labels)

    cnt = np.zeros(nL); s0 = np.zeros(nL); s1 = np.zeros(nL); s2 = np.zeros(nL)
    d1, d2 = ann.shape[1], ann.shape[2]
    g1 = np.repeat(np.arange(d1), d2).astype(float)
    g2 = np.tile(np.arange(d2), d1).astype(float)
    for i0 in range(ann.shape[0]):
        flat = lut[ann[i0].ravel()]
        cnt += np.bincount(flat, minlength=nL)
        s0  += np.bincount(flat, weights=np.full(flat.size, float(i0)), minlength=nL)
        s1  += np.bincount(flat, weights=g1, minlength=nL)
        s2  += np.bincount(flat, weights=g2, minlength=nL)

    id_to_row = {int(l): int(lut[int(l)]) for l in labels}
    centroids = np.full((len(acronyms), 3), np.nan)
    unmapped = []
    for i, a in enumerate(acronyms):
        try:
            sid = bg_atlas.structures[a]["id"]
        except KeyError:
            unmapped.append(a); continue
        # own id + every descendant, so a non-leaf acronym aggregates correctly
        ids = [v["id"] for v in bg_atlas.structures.values()
               if sid in v["structure_id_path"]]
        rows = [id_to_row[j] for j in ids if j in id_to_row]
        if not rows or cnt[rows].sum() == 0:
            unmapped.append(a); continue
        tot = cnt[rows].sum()
        centroids[i] = np.array([s0[rows].sum() / tot * res_um[0],
                                 s1[rows].sum() / tot * res_um[1],
                                 s2[rows].sum() / tot * res_um[2]])

    if unmapped:
        print(f"WARNING: {len(unmapped)} acronym(s) had no voxels / no atlas entry and got "
              f"NaN centroids -- excluded from spatial draws: {unmapped[:15]}"
              f"{' ...' if len(unmapped) > 15 else ''}")

    cache.parent.mkdir(parents=True, exist_ok=True)
    np.savez(cache, centroids=centroids, acronyms=np.array(list(acronyms), dtype=object))
    print(f"Built and cached centroids for {len(acronyms)} regions -> {cache}")
    return centroids


CENTROIDS = build_region_centroids(PERM_REGIONS)
print(f"{np.isfinite(CENTROIDS).all(axis=1).sum()} / {len(PERM_REGIONS)} regions have "
      f"usable centroids for spatial draws")

### 12e. Region null: size-matched resampling

Holds the observed per-region statistics fixed and asks whether *this anatomy* is
special relative to size-matched alternatives.

- **`"spatial"`** (default) -- size-matched *and* spatially contiguous draws:
  random seed region, grown to `k` nearest neighbours by CCF centroid distance.
  This is the discrete-parcellation analogue of a spin test.
- **`"random"`** -- uniform size-matched draws. Reported alongside, but it does
  **not** control for spatial autocorrelation: an anatomically contiguous subset
  drawn from a spatially smooth effect map will beat scattered random sets for
  reasons that have nothing to do with the biology. Since MA/NDB/SI is
  contiguous, `"random"` flatters it, and `"spatial"` is the honest number.

Reporting both is the point: if a set survives the spatial null too, contiguity
is not the explanation.

In [ ]:
def region_null_sets(k, n_draw, mode="spatial", centroids=None, seed=0, candidate_mask=None):
    """Yield n_draw boolean masks over PERM_REGIONS, each selecting k regions."""
    rng = np.random.default_rng(seed)
    n_regions = len(PERM_REGIONS)
    cand = np.flatnonzero(candidate_mask) if candidate_mask is not None else np.arange(n_regions)

    if mode == "random":
        for _ in range(n_draw):
            m = np.zeros(n_regions, dtype=bool)
            m[rng.choice(cand, size=k, replace=False)] = True
            yield m

    elif mode == "spatial":
        if centroids is None:
            raise ValueError("mode='spatial' needs centroids -- run build_region_centroids() (12d).")
        ok = np.isfinite(centroids[cand]).all(axis=1)
        cand = cand[ok]
        if len(cand) < k:
            raise ValueError(f"Only {len(cand)} regions have usable centroids; need >= {k}.")
        C = centroids[cand]
        D = np.linalg.norm(C[:, None, :] - C[None, :, :], axis=-1)
        order = np.argsort(D, axis=1)     # self first, then nearest outward
        for _ in range(n_draw):
            seed_i = rng.integers(len(cand))
            m = np.zeros(n_regions, dtype=bool)
            m[cand[order[seed_i, :k]]] = True
            yield m
    else:
        raise ValueError(mode)


def enrichment_region_null(res, acronyms, stat="mean_z_diff", tail=None, mode="spatial",
                           n_draw=5000, centroids=None, seed=0, label=None,
                           ax=None, plot=True, bins=45):
    label = label or f"{len(acronyms)} regions"
    tail = tail or default_tail(stat)
    in_mask, present, missing = resolve_regions(acronyms, res, label=label)
    if in_mask.sum() == 0:
        print(f"  ! {label}: no regions resolved -- skipping"); return None

    # Only regions with a usable observed statistic can be drawn, or a draw would
    # quietly contribute NaN and shrink below size k.
    #
    # ### FIX ###
    # The requested set must be intersected with `usable` too. Null draws are
    # all-usable by construction, so leaving unusable regions in `in_mask` made
    # the observed statistic an average over FEWER effective regions than the
    # size-matched null it was compared against -- a size mismatch that biases
    # the comparison in an uncontrolled direction.
    usable = np.isfinite(res.obs_z)
    n_before = int(in_mask.sum())
    in_mask = in_mask & usable
    k = int(in_mask.sum())
    if k == 0:
        print(f"  ! {label}: no regions with a usable observed statistic -- skipping")
        return None
    if k < n_before:
        print(f"  ! {label}: {n_before - k} region(s) dropped for a non-finite observed "
              f"z; drawing size-matched sets of k={k}")

    obs = subset_statistic(res.obs_neglog10p, res.obs_z, in_mask, stat)
    null = np.array([subset_statistic(res.obs_neglog10p, res.obs_z, m, stat)
                     for m in region_null_sets(k, n_draw, mode=mode, centroids=centroids,
                                               seed=seed, candidate_mask=usable)])
    pval = perm_pvalue(null, obs, tail=tail)

    if plot:
        created = ax is None
        if created:
            _, ax = plt.subplots(figsize=(6, 4.2))
        ax.hist(null, bins=bins, color="#8172B2", alpha=0.85, edgecolor="white")
        ax.axvline(obs, color="#C44E52", lw=2.2, label=f"observed = {obs:.3f}")
        ax.axvline(np.nanmean(null), color="gray", lw=1, ls=":", label="null mean")
        ax.set(xlabel=stat, ylabel=f"{mode} size-matched region sets",
               title=f"{label} (n={k})\n{res.contrast} | {mode.upper()} region null | "
                     f"p = {pval:.4f}")
        ax.legend(fontsize=8)
        if created:
            plt.tight_layout()

    partner = VS_ALL_PARTNER.get(stat)
    obs_partner = (subset_statistic(res.obs_neglog10p, res.obs_z, in_mask, partner)
                   if partner else np.nan)

    return {"set": label, "contrast": res.contrast, "null_type": f"region:{mode}",
            "stat": stat, "n_regions": k, "n_missing": len(missing), "observed": obs,
            "observed_vs_complement": obs_partner, "frac_of_brain": k / len(res.regions),
            "null_mean": np.nanmean(null), "null_sd": np.nanstd(null),
            "z_vs_null": (obs - np.nanmean(null)) / np.nanstd(null),
            "perm_p": pval, "tail": tail, "n_draws": n_draw}

### 12f. Run everything

In [ ]:
# ### CHOICE ### -- which contrast's null to test enrichment against
ENRICH_CONTRAST = "geno:sex:light"
#ENRICH_CONTRAST = "light (main effect)"
#ENRICH_CONTRAST ="geno:sex | dark"
res = perm_results[ENRICH_CONTRAST]

# ### CHANGED ### subset vs WHOLE BRAIN. Identical p to "mean_z_diff"; see the
# Section 12 header and the assertion in 12b. The complement-form effect size is
# carried alongside in `observed_vs_complement`.
ENRICH_STAT = "mean_z_vs_all"

testable = [k for k, v in REGION_SETS.items() if sum(a in REGION_POS for a in v) >= 2]
rows = []


def run_panel(fn, title, fname, set_dict=None, names=None, **kw):
    set_dict = set_dict if set_dict is not None else REGION_SETS
    names = names if names is not None else testable
    print("=" * 70); print(title); print("=" * 70)
    ncols = min(len(names), 5)
    nrows = int(np.ceil(len(names) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.4 * ncols, 4.3 * nrows),
                             squeeze=False)
    flat = axes.reshape(-1)
    out = []
    for ax, name in zip(flat, names):
        r = fn(res, set_dict[name], label=name, ax=ax, **kw)
        if r:
            out.append(r)
    for ax in flat[len(names):]:
        ax.axis("off")
    plt.tight_layout(); plt.savefig(OUT_DIR / fname, dpi=200); plt.show()
    return out


# --- 1) region null, SPATIAL: the primary enrichment test ---
rows += run_panel(enrichment_region_null, "REGION NULL, SPATIAL (primary)",
                  "enrichment_region_null_spatial.png",
                  stat=ENRICH_STAT, mode="spatial", n_draw=5000, centroids=CENTROIDS)

# --- 2) region null, RANDOM: reported alongside; flatters contiguous sets ---
rows += run_panel(enrichment_region_null, "REGION NULL, RANDOM (comparison)",
                  "enrichment_region_null_random.png",
                  stat=ENRICH_STAT, mode="random", n_draw=5000)

# --- 3) label null: the complement that respects inter-regional correlation ---
rows += run_panel(enrichment_label_null, "LABEL NULL (complement)",
                  "enrichment_label_null.png", stat=ENRICH_STAT)

In [ ]:
enrichment_table = pd.DataFrame(rows)

# ### CHOICE ### -- multiplicity across sets
# BH across `perm_p` within each null type, reported alongside the raw p. The
# Allen subdivisions in Section 12x are corrected as a SEPARATE family: they are
# an exhaustive partition of the brain, whereas these sets are hypothesis-driven,
# and pooling them would let the partition dilute the pre-specified tests.
enrichment_table["q_bh"] = np.nan
for nt, sub in enrichment_table.groupby("null_type"):
    ok = sub["perm_p"].notna()
    if ok.sum():
        _, q, _, _ = multipletests(sub.loc[ok, "perm_p"], method="fdr_bh")
        enrichment_table.loc[sub.index[ok], "q_bh"] = q

print()
print(enrichment_table.to_string(index=False))
enrichment_table.to_csv(OUT_DIR / "enrichment_permutation_results.csv", index=False)

### 12g. Reporting notes

- **Lead the enrichment claim with the spatial region null, report the others
  alongside.** "Concentrated in S relative to spatially matched region sets
  (p = ..), and the S-vs-background gap exceeds the label-permutation null
  (p = ..)" is defensible. Leading with the label null alone is the thing to
  avoid -- see the simulation table above.
- **Say which permutation scheme each number came from.** Simple effects use
  exact label permutation; interaction terms use Freedman-Lane, which is
  approximate. Both belong in the methods, distinguished.
- **Report the resolution ceiling** for label-permutation contrasts. If there
  are only 126 distinct labellings, `p = .008` is the floor -- say so rather
  than writing `p < .001`.
- **`mean_z_diff` is the enrichment statistic; `sum_in` is not.** If `sum_in`
  appears anywhere, frame it as "there is an effect within S", never as "S is
  enriched".
- **Signed vs unsigned disagreement is a finding.** A large `sum(-log10 p)`
  with `mean_z` near zero means many regions responding in inconsistent
  directions -- spatial redistribution, not brainwide gain. That maps onto the
  PLSC scalar-gate versus multivariate-gate distinction and should be reported
  as such, not smoothed over.
- **`n = 1` sets cannot be enrichment-tested.** MS alone has no distribution; it
  is a comparator, and should be reported as one rather than enrichment-tested
  on its own.
- **The split two-ways are a planned decomposition, not new tests.** Frame
  `sex:light | control` and `sex:light | MKO` as the licensed readout of the
  three-way gate, and keep the confirmatory/descriptive tables separate as
  elsewhere in this project.
- **The enrichment statistic is subset vs. the WHOLE BRAIN**
  (`mean(z | in S) - mean(z | all)`), not subset vs. complement. It differs from
  the complement form by the constant `1 - k/N` and yields an identical
  permutation p, so describe the switch as a change of reported effect size, never
  as a re-analysis. Both columns are in the output table.
- **Allen subdivisions (12x) are a screen, not a confirmatory test.** They are BH
  corrected as their own family, the random region null is not reported for them
  because a division is contiguous by construction, and divisions covering a large
  share of the brain are flagged as low-resolution.
- **The region set is pre-committed.** `REQUIRE_COMPLETE=True` restricts every
  enrichment claim to regions with complete data across all mice. That is the
  analysis; there is no relaxed variant to fall back on.

## 12x. Enrichment by Allen major subdivision

The same machinery, applied to every major CCFv3 division rather than to
hypothesis-driven sets. This answers "is the three-way concentrated anywhere in
the coarse anatomy" without requiring a prior about where.

### CHOICE -- spatial null only

A major division **is** a contiguous anatomical block. The random region null
draws scattered sets, so on a spatially smooth effect map it will flag almost
every division as enriched for reasons that have nothing to do with the biology
-- the exact failure mode the spatial null exists to prevent (Fulcher et al.
2021). The random null is therefore **not reported here**. The label null is
kept, because it randomizes design labels rather than region identity and so is
not subject to the contiguity problem.

### FLAG -- large divisions have no resolution

For a division covering a large share of the brain, "in S vs the whole brain" is
close to comparing the map against itself, and the spatial null cannot draw a
size-matched contiguous block that is not mostly the same tissue. Divisions above
`FRAC_BRAIN_WARN` are computed but flagged, and should be read as descriptive.
This is the main reason the subdivision table is a screen and not a confirmatory
analysis.

### CHOICE -- separate BH family

The divisions partition the brain exhaustively; the Section 12 sets are chosen
from prior anatomy. Correcting them together would let a 14-member partition
dilute the pre-specified tests. They are separate families and are labelled as
such in the output.

In [ ]:
# Subdivision sets are built from the SAME `region_to_major` mapping used in
# Section 9, so coverage counts and enrichment rows refer to identical anatomy.
FRAC_BRAIN_WARN = 0.25     # above this share of regions, flag as low-resolution
MIN_DIVISION_N  = 5        # below this many regions, the null has nothing to work with

DIVISION_SETS = {}
for _m in MAJOR_STRUCTURES:
    members = [a for a in PERM_REGIONS if region_to_major.get(a) == _m]
    if len(members) >= MIN_DIVISION_N:
        DIVISION_SETS[_m] = members

_unassigned = [a for a in PERM_REGIONS if region_to_major.get(a) is None]
print(f"{len(DIVISION_SETS)} of {len(MAJOR_STRUCTURES)} divisions have "
      f">= {MIN_DIVISION_N} regions in the permutation set")
for _m, mem in DIVISION_SETS.items():
    frac = len(mem) / len(PERM_REGIONS)
    flag = "   << large: low resolution" if frac > FRAC_BRAIN_WARN else ""
    print(f"  {_m:<14} {len(mem):>4} regions  ({100*frac:4.1f}% of brain){flag}")
if _unassigned:
    print(f"  {len(_unassigned)} region(s) unassigned to any division, excluded: {_unassigned[:10]}")
_skipped = [m for m in MAJOR_STRUCTURES if m not in DIVISION_SETS]
if _skipped:
    print(f"  skipped (< {MIN_DIVISION_N} regions): {_skipped}")

In [ ]:
div_rows = []

# --- primary: spatial region null ---
div_rows += run_panel(enrichment_region_null,
                      f"SUBDIVISION ENRICHMENT -- SPATIAL REGION NULL (primary)  [{ENRICH_CONTRAST}]",
                      "enrichment_divisions_spatial.png",
                      set_dict=DIVISION_SETS, names=list(DIVISION_SETS),
                      stat=ENRICH_STAT, mode="spatial", n_draw=5000, centroids=CENTROIDS)

# --- complement: label null ---
div_rows += run_panel(enrichment_label_null,
                      f"SUBDIVISION ENRICHMENT -- LABEL NULL  [{ENRICH_CONTRAST}]",
                      "enrichment_divisions_label.png",
                      set_dict=DIVISION_SETS, names=list(DIVISION_SETS),
                      stat=ENRICH_STAT)

# ### NOT RUN ### the random region null. See the section header: a division is
# contiguous by construction, so the random null is not a valid comparison for it.

division_table = pd.DataFrame(div_rows)
division_table["family"] = "allen_major_division"

# BH within null type, as a SEPARATE family from the pre-specified sets.
division_table["q_bh"] = np.nan
for nt, sub in division_table.groupby("null_type"):
    ok = sub["perm_p"].notna()
    if ok.sum():
        _, q, _, _ = multipletests(sub.loc[ok, "perm_p"], method="fdr_bh")
        division_table.loc[sub.index[ok], "q_bh"] = q

division_table["low_resolution"] = division_table["frac_of_brain"] > FRAC_BRAIN_WARN

# ---- one row per division, both nulls side by side ----
_piv = division_table.pivot_table(index="set", columns="null_type",
                                  values=["perm_p", "q_bh"])
_piv.columns = [f"{a}_{b}" for a, b in _piv.columns]
_meta = (division_table.drop_duplicates("set").set_index("set")
         [["n_regions", "frac_of_brain", "observed", "observed_vs_complement",
           "z_vs_null", "low_resolution"]])
division_summary = (_meta.join(_piv)
                    .reindex([m for m in MAJOR_STRUCTURES if m in _meta.index])
                    .round(4))

print("=" * 110)
print(f"ENRICHMENT BY ALLEN MAJOR DIVISION  |  {ENRICH_CONTRAST}  |  {ENRICH_STAT}")
print("=" * 110)
print(division_summary.to_string())
print(f"\n  observed               = mean(z | in division) - mean(z | all regions)")
print(f"  observed_vs_complement = the same quantity against the complement instead")
print(f"                           (identical p; differs by the factor 1 - k/N)")
print(f"  z_vs_null              = (observed - spatial null mean) / spatial null SE")
print(f"  low_resolution         = division covers > {100*FRAC_BRAIN_WARN:.0f}% of regions;")
print(f"                           read as descriptive, not as a test")
print("\n  BH correction is WITHIN this family only -- these are not pooled with the")
print("  pre-specified sets in Section 12.")

division_table.to_csv(OUT_DIR / "enrichment_by_division.csv", index=False)
division_summary.to_csv(OUT_DIR / "enrichment_by_division_summary.csv")

In [ ]:
print("=" * 78)
print("DESIGN AND POWER AUDIT")
print("=" * 78)

# ---- cell counts ----
cell_n = {cell: int(n) for cell, n in zip(CELLS, np.bincount(OBS_CELL_IDX, minlength=8))}
print(f"\n{len(mouse_design)} mice across 8 design cells:")
cell_tab = pd.DataFrame([{"genotype": g, "sex": s, "light": l, "n": cell_n[(g, s, l)]}
                         for g, s, l in CELLS])
print(cell_tab.to_string(index=False))
print(f"\n  min cell n = {min(cell_n.values())}   max = {max(cell_n.values())}   "
      f"balanced would be {len(mouse_design)/8:.1f}")

thin = {c: n for c, n in cell_n.items() if n < UNDERPOWERED_CELL_N}
if thin:
    print(f"\n  {len(thin)} cell(s) below UNDERPOWERED_CELL_N={UNDERPOWERED_CELL_N}:")
    for c, n in thin.items():
        print(f"    {c}: n={n}")

# ---- which FDR families are populated ----
fam = (all_results.groupby("contrast")
       .agg(n_tested=("p", "size"),
            n_powered=("underpowered", lambda x: int((~x).sum())),
            min_cell_n=("min_cell_n", "min"))
       .reindex(list(ALL_CONTRASTS)))
fam["q_powered_usable"] = np.where(fam["n_powered"] > 0, "yes", "EMPTY FAMILY")
fam["n_sig_q_all"] = [int((all_results.loc[all_results.contrast == c, "q_all"] < 0.05).sum())
                      for c in fam.index]
fam["n_sig_q_powered"] = [int((all_results.loc[all_results.contrast == c, "q_powered"] < 0.05).sum())
                          for c in fam.index]

n_empty = (fam["n_powered"] == 0).sum()
print(f"\n{n_empty} / {len(fam)} contrast families have ZERO well-powered rows.")
if n_empty:
    print("For those, n_sig_q_powered = 0 means the family was empty, NOT that")
    print("nothing survived correction. Do not report those two numbers side by side.")
print()
print(fam.to_string())

# ---- region-set bookkeeping ----
print(f"\nRegion sets in play (these are NOT the same brain -- do not cross-quote counts):")
print(f"  fitted / FDR analysis   : {all_results['Acronym'].nunique()} regions")
print(f"  permutation + enrichment: {len(PERM_REGIONS)} regions "
      f"(REQUIRE_COMPLETE={REQUIRE_COMPLETE})")
only_fdr = sorted(set(all_results['Acronym'].unique()) - set(PERM_REGIONS))
print(f"  in FDR set but not permutation set: {len(only_fdr)}")
if only_fdr:
    maj = pd.Series([region_to_major.get(a) for a in only_fdr]).value_counts()
    print(f"    by major structure: {dict(maj)}")

power_audit = fam

In [ ]:
print("=" * 78)
print("PER-MOUSE GLOBAL GAIN")
print("=" * 78)

# Whole-brain mean per mouse, over the COMMON region set so every mouse is
# averaged over identical anatomy (a plain groupby over `d` would average each
# mouse over whatever regions it happens to have, confounding gain with coverage).
gain = np.nanmean(XMAT, axis=0)                     # (n_mice,)
gain_s = pd.Series(gain, index=wide.columns, name="whole_brain_mean_log_density")

gain_df = mouse_design.copy()
gain_df["whole_brain_mean"] = gain_s
batch_lookup = d[["mouse_id", "batch"]].drop_duplicates().set_index("mouse_id")["batch"]
gain_df["batch"] = batch_lookup.reindex(gain_df.index)
gain_df = gain_df.sort_values(["genotype", "sex", "light", "whole_brain_mean"])

print(f"\nPer-mouse whole-brain mean log density (over the {len(PERM_REGIONS)} "
      f"complete regions):\n")
print(gain_df.to_string())

per_cell = (gain_df.groupby(["genotype", "sex", "light"], observed=True)["whole_brain_mean"]
            .agg(n="size", mean="mean", sd="std", lo="min", hi="max").round(3))
print(f"\nBy design cell:\n")
print(per_cell.to_string())

overall_sd = gain_df["whole_brain_mean"].std()
within_sd = (gain_df.groupby(["genotype", "sex", "light"], observed=True)["whole_brain_mean"]
             .std().mean())
between_sd = per_cell["mean"].std()


def eta_squared(values, groups):
    """Fraction of variance in `values` explained by `groups`."""
    v = pd.Series(np.asarray(values, dtype=float))
    g = pd.Series(np.asarray(groups))
    grand = v.mean()
    ss_total = ((v - grand) ** 2).sum()
    ss_between = sum(len(idx) * (v[idx.values].mean() - grand) ** 2
                     for _, idx in v.groupby(g.values).groups.items()
                     for idx in [pd.Index(idx)])
    return ss_between / ss_total if ss_total > 0 else np.nan


cell_key = gain_df[["genotype", "sex", "light"]].astype(str).agg("|".join, axis=1)
eta_cell = eta_squared(gain_df["whole_brain_mean"], cell_key)
eta_batch = eta_squared(gain_df["whole_brain_mean"], gain_df["batch"])

print(f"\n  SD across all mice          : {overall_sd:.3f} log units "
      f"({np.exp(overall_sd):.1f}x in density)")
print(f"  mean SD WITHIN design cells : {within_sd:.3f}")
print(f"  SD of cell MEANS            : {between_sd:.3f}")
print(f"\n  variance in whole-brain gain explained by DESIGN CELL : {eta_cell:.1%}")
print(f"  variance in whole-brain gain explained by IMMUNO BATCH: {eta_batch:.1%}")
print("\n  ### HOW TO READ THIS ###")
print("  A large design-cell share is NOT by itself evidence of an artifact -- a")
print("  genuine brainwide effect would produce exactly the same thing. What")
print("  distinguishes them is the batch column and the crosstabs below:")
print("    cell high, batch low, batch balanced across groups  -> consistent with real biology")
print("    batch high, and batch confounded with genotype/sex/light")
print("                                                        -> gain is technical and")
print("                                                           inseparable from group by")
print("                                                           any permutation test")
print("  If the crosstabs show batch nested within group, no reanalysis of THIS")
print("  dataset can settle it -- that requires design information or a rebalanced cohort.")

print(f"\nGain vs immuno batch (is gain a staining property?):")
print(gain_df.groupby("batch")["whole_brain_mean"].agg(n="size", mean="mean", sd="std").round(3).to_string())
print(f"\nBatch x genotype crosstab (is batch confounded with group?):")
print(pd.crosstab(gain_df["batch"], gain_df["genotype"]).to_string())
print(f"\nBatch x light crosstab:")
print(pd.crosstab(gain_df["batch"], gain_df["light"]).to_string())

In [ ]:
# ---- How much of each contrast is explained by cell-level gain alone? ----
gain_cell_means = np.array([gain[OBS_CELL_IDX == c].mean() for c in range(8)])

gain_rows = []
for cname, (ctype, w) in ALL_CONTRASTS.items():
    sub = all_results[all_results.contrast == cname]
    if len(sub) == 0:
        continue
    predicted = float(w @ gain_cell_means)     # offset from gain alone
    observed = float(sub["log_estimate"].median())
    same_sign = float((np.sign(sub["log_estimate"]) == np.sign(predicted)).mean()) \
        if predicted != 0 else np.nan
    gain_rows.append({
        "contrast": cname, "contrast_type": ctype,
        "gain_predicted": round(predicted, 3),
        "median_observed": round(observed, 3),
        "obs_minus_pred": round(observed - predicted, 3),
        "frac_regions_matching_predicted_sign": round(same_sign, 3),
        "n_regions": len(sub),
    })
gain_explains = pd.DataFrame(gain_rows).set_index("contrast").reindex(list(ALL_CONTRASTS)).dropna(how="all")

print("=" * 78)
print("HOW MUCH OF EACH CONTRAST IS CELL-LEVEL GAIN?")
print("=" * 78)
print("gain_predicted = w . (per-cell mean of per-mouse whole-brain means)")
print("If gain_predicted ~ median_observed AND nearly every region shares its sign,")
print("that contrast is a brainwide offset rather than a regional effect.\n")
print(gain_explains.to_string())

ok = gain_explains[["gain_predicted", "median_observed"]].dropna()
if len(ok) > 2:
    r = np.corrcoef(ok["gain_predicted"], ok["median_observed"])[0, 1]
    slope = np.polyfit(ok["gain_predicted"], ok["median_observed"], 1)[0]
    print(f"\nAcross the {len(ok)} contrasts: corr(predicted, observed) = {r:.3f}, "
          f"slope = {slope:.3f}")
    print("  corr near 1 with slope near 1 means the per-region estimates are, to first")
    print("  order, a readout of animal-level gain projected through the contrast weights.")

flagged = gain_explains[gain_explains["frac_regions_matching_predicted_sign"] > 0.95]
if len(flagged):
    print(f"\n!! {len(flagged)} contrast(s) have >95% of regions on the gain-predicted side")
    print("   of zero -- a whole-brain translation, not a mixture of null and signal:")
    for c in flagged.index:
        print(f"     {c}")

In [ ]:
print("=" * 78)
print("GLOBAL BREADTH")
print("=" * 78)

g = global_perm_table.copy()
wide_g = g.pivot(index="contrast", columns="stat",
                 values=["observed", "perm_p"])
wide_g.columns = [f"{b}_{a}" for a, b in wide_g.columns]
meta = g.drop_duplicates("contrast").set_index("contrast")[["method", "resolution_floor", "n_perm"]]
breadth = meta.join(wide_g).reindex([c for c in PERM_CONTRASTS])

breadth["p_over_floor"] = (breadth["mean_z_perm_p"] / breadth["resolution_floor"]).replace(np.inf, np.nan)
breadth["at_resolution_limit"] = breadth["p_over_floor"] < 3

breadth = breadth[["method", "mean_z_observed", "mean_z_perm_p",
                   "sum_observed", "sum_perm_p", "resolution_floor",
                   "at_resolution_limit"]].round(4)
print()
print(breadth.to_string())

print("\nReading guide:")
print("  mean_z  significant + sum significant -> consistent brainwide shift in one direction")
print("  mean_z  n.s.        + sum significant -> many regions responding in MIXED directions")
print("                                           (spatial redistribution, not gain)")
print("  at_resolution_limit=True -> report the floor alongside p; more permutations")
print("                              cannot lower it.")

sig = breadth[breadth["mean_z_perm_p"] < 0.05]
print(f"\n{len(sig)} / {len(breadth)} contrasts significant on signed mean_z at 0.05:")
for c in sig.index:
    print(f"  {c}: mean_z = {sig.loc[c,'mean_z_observed']:+.3f}, "
          f"p = {sig.loc[c,'mean_z_perm_p']:.4f} ({sig.loc[c,'method']})")

breadth_summary = breadth

In [ ]:
print("=" * 100)
print(f"ENRICHMENT  |  contrast: {ENRICH_CONTRAST}  |  statistic: {ENRICH_STAT}")
print("=" * 100)

e = enrichment_table.copy()

# p-value from each null, one column per null type
piv = e.pivot_table(index="set", columns="null_type", values="perm_p")

# Observed statistic and set bookkeeping (identical across null types for a
# given set -- only the null changes, not the data)
_obs_cols = [c for c in ["observed", "observed_vs_complement", "n_regions",
                         "n_missing", "frac_of_brain"] if c in e.columns]
obs = e.drop_duplicates("set").set_index("set")[_obs_cols]

# ---- Spatial null moments: the estimate and its permutation standard error ----
# null_mean is where the statistic sits under size- and contiguity-matched
# resampling (the expected value if this set were anatomically unremarkable);
# null_sd is the permutation standard error of the statistic, i.e. the scale
# against which `observed` should be read. spatial_z = (observed - null_mean) /
# null_sd is the effect size in SE units -- report it alongside p, because a
# permutation p is floored by the number of draws while the z is not.
PRIMARY_NULL = "region:spatial"
if PRIMARY_NULL in set(e["null_type"]):
    spatial = (e[e["null_type"] == PRIMARY_NULL]
               .drop_duplicates("set")
               .set_index("set")[["null_mean", "null_sd", "z_vs_null", "n_draws"]]
               .rename(columns={"null_mean": "sp_null_mean",
                                "null_sd":   "sp_null_se",
                                "z_vs_null": "sp_z",
                                "n_draws":   "sp_n_draws"}))
    # Monte Carlo SE of the p-value itself -- with a finite number of draws the
    # p-value is an estimate too, and a p of 0.024 from 5000 draws carries about
    # +-0.002. Worth quoting when a p sits near 0.05.
    sp_p = (e[e["null_type"] == PRIMARY_NULL].drop_duplicates("set")
            .set_index("set")["perm_p"].astype(float))
    spatial["sp_p_mc_se"] = np.sqrt(sp_p * (1 - sp_p) / spatial["sp_n_draws"])
else:
    spatial = pd.DataFrame(index=obs.index,
                           columns=["sp_null_mean", "sp_null_se", "sp_z",
                                    "sp_n_draws", "sp_p_mc_se"], dtype=float)
    print(f"\n!! No '{PRIMARY_NULL}' rows in enrichment_table -- the primary null did not run.")

enrich = obs.join(spatial).join(piv).round(4)

col_spatial = PRIMARY_NULL if PRIMARY_NULL in enrich.columns else None
col_random  = "region:random" if "region:random" in enrich.columns else None
col_label   = "label" if "label" in enrich.columns else None


def verdict(r):
    if col_spatial is None or pd.isna(r.get(col_spatial)):
        return "no spatial null"
    if r[col_spatial] < 0.05:
        others = [c for c in (col_random, col_label) if c and pd.notna(r.get(c))]
        if all(r[c] < 0.05 for c in others):
            return "ENRICHED (all nulls)"
        return "enriched (spatial; check others)"
    if col_random and pd.notna(r.get(col_random)) and r[col_random] < 0.05:
        return "contiguity artifact (random only)"
    return "not enriched"


enrich["direction"] = np.where(enrich["observed"] > 0, "enriched", "depleted")
enrich["verdict"] = enrich.apply(verdict, axis=1)

# Column order: what was measured, then the primary null's moments, then every
# null's p, then the call
col_order = ([c for c in ["n_regions", "n_missing", "frac_of_brain", "observed",
                          "observed_vs_complement",
                          "sp_null_mean", "sp_null_se", "sp_z", "sp_p_mc_se"]
              if c in enrich.columns]
             + [c for c in (col_spatial, col_random, col_label) if c]
             + ["direction", "verdict"])
order = [k for k in REGION_SETS if k in enrich.index]
enrich = enrich.reindex(order)[col_order]

print()
print(enrich.to_string())
_desc = ("mean(z | in set) - mean(z | ALL regions)" if ENRICH_STAT.endswith("_vs_all")
         else "mean(z | in set) - mean(z | outside set)")
print(f"\n  observed     = {_desc}")
if ENRICH_STAT.endswith("_vs_all"):
    print("  (the complement form is in `observed_vs_complement`; it differs by the")
    print("   constant 1 - k/N and yields an IDENTICAL permutation p)")
print(f"  sp_null_mean = same statistic under {PRIMARY_NULL} resampling (expected ~0)")
print(f"  sp_null_se   = permutation SE of the statistic under {PRIMARY_NULL}")
print("  sp_z         = (observed - sp_null_mean) / sp_null_se")
print("  sp_p_mc_se   = Monte Carlo SE of the spatial p itself, from a finite n_draws")

if (enrich["n_missing"] > 0).any():
    print("\n!! Some sets ran short of their definition:")
    for s in enrich.index[enrich["n_missing"] > 0]:
        got, miss = int(enrich.loc[s, "n_regions"]), int(enrich.loc[s, "n_missing"])
        print(f"   {s}: {got} of {got+miss} acronyms resolved -- fix or state this")

near = enrich[(enrich[col_spatial] - 0.05).abs() < 2 * enrich["sp_p_mc_se"]] \
    if col_spatial and "sp_p_mc_se" in enrich else enrich.iloc[0:0]
if len(near):
    print("\n!! Spatial p within 2 Monte Carlo SEs of 0.05 -- the call is not stable at")
    print(f"   this n_draws; raise it before quoting a verdict for: {list(near.index)}")

enrichment_summary = enrich

In [ ]:
print("=" * 78)
print("ROSTRO-CAUDAL POSITION CHECK")
print("=" * 78)

AP = CENTROIDS[:, 0]      # axis 0 of the CCF annotation is anterior -> posterior, in um
finite = np.isfinite(AP) & np.isfinite(res.obs_z)
print(f"\n{finite.sum()} / {len(PERM_REGIONS)} regions have both a centroid and a usable z")

r_ap = np.corrcoef(AP[finite], res.obs_z[finite])[0, 1]
r_ap_abs = np.corrcoef(AP[finite], np.abs(res.obs_z[finite]))[0, 1]
print(f"corr(A-P position, signed z)   = {r_ap:+.3f}")
print(f"corr(A-P position, |z|)        = {r_ap_abs:+.3f}   "
      f"(a strong positive value = caudal regions carry more effect)")

# ---- effect by major structure, roughly rostral -> caudal ----
maj = pd.Series([region_to_major.get(a) for a in PERM_REGIONS], index=PERM_REGIONS)
by_struct = (pd.DataFrame({"major": maj.values, "z": res.obs_z, "ap_um": AP})
             .dropna(subset=["major"])
             .groupby("major")
             .agg(n=("z", "size"), mean_z=("z", "mean"), mean_absz=("z", lambda x: np.abs(x).mean()),
                  mean_ap_um=("ap_um", "mean"))
             .reindex(MAJOR_STRUCTURES).dropna(how="all").round(3))
print(f"\nEffect by Allen major division ({ENRICH_CONTRAST}):\n")
print(by_struct.to_string())

# ---- where each region set sits on the A-P axis ----
print(f"\nMean A-P position of each region set (brain spans "
      f"{np.nanmin(AP):.0f}-{np.nanmax(AP):.0f} um):\n")
for name, acrs in REGION_SETS.items():
    m, _, _ = resolve_regions(acrs, res, warn=False)
    if m.sum():
        print(f"  {name:<32} n={m.sum():>3}   mean A-P = {np.nanmean(AP[m]):>7.0f} um")
print(f"  {'(all permutation regions)':<32} n={len(PERM_REGIONS):>3}   "
      f"mean A-P = {np.nanmean(AP):>7.0f} um")

In [ ]:
print("=" * 78)
print("READOUT")
print("=" * 78)

print(f"\nDESIGN")
print(f"  {len(mouse_design)} mice, cells n = "
      f"{', '.join(str(cell_n[c]) for c in CELLS)} (min {min(cell_n.values())})")
print(f"  {all_results['Acronym'].nunique()} regions fitted; "
      f"{len(PERM_REGIONS)} used for permutation and enrichment")
print(f"  {int((power_audit['n_powered'] == 0).sum())} of {len(power_audit)} "
      f"contrast families have no well-powered rows")

print(f"\nGLOBAL GAIN")
print(f"  per-mouse whole-brain mean spans {gain.min():.2f} to {gain.max():.2f} log units "
      f"({np.exp(gain.max()-gain.min()):.0f}x in density)")
print(f"  gain variance explained by design cell = {eta_cell:.1%}, "
      f"by immuno batch = {eta_batch:.1%}")
n_flag = int((gain_explains['frac_regions_matching_predicted_sign'] > 0.95).sum())
print(f"  {n_flag} contrast(s) with >95% of regions on the gain-predicted side of zero")

print(f"\nGLOBAL BREADTH (signed mean z, permutation)")
for c in breadth_summary.index:
    r_ = breadth_summary.loc[c]
    flag = "  [at resolution limit]" if r_["at_resolution_limit"] else ""
    star = " *" if r_["mean_z_perm_p"] < 0.05 else "  "
    print(f" {star} {c:<32} mean_z = {r_['mean_z_observed']:+.3f}   "
          f"p = {r_['mean_z_perm_p']:.4f}   ({r_['method']}){flag}")

print(f"\nENRICHMENT ({ENRICH_CONTRAST}, {ENRICH_STAT})")
for s in enrichment_summary.index:
    r_ = enrichment_summary.loc[s]
    ps = "  ".join(f"{c.split(':')[-1]}={r_[c]:.3f}" for c in
                   (col_spatial, col_random, col_label) if c and pd.notna(r_.get(c)))
    print(f"  {s:<32} {r_['observed']:+.3f} ({r_['direction']})   {ps}")
    print(f"  {'':<32} -> {r_['verdict']}")

print(f"\nENRICHMENT BY ALLEN DIVISION ({ENRICH_CONTRAST}) -- screen, separate BH family")
_dcol = [c for c in division_summary.columns if c.startswith("perm_p_region:spatial")]
for s_ in division_summary.index:
    r_ = division_summary.loc[s_]
    _p = r_[_dcol[0]] if _dcol else np.nan
    _flag = "  [low resolution]" if r_.get("low_resolution") else ""
    _star = " *" if (pd.notna(_p) and _p < 0.05) else "  "
    print(f" {_star} {s_:<14} n={int(r_['n_regions']):>3}  "
          f"obs = {r_['observed']:+.3f}  spatial p = {_p:.3f}{_flag}")

if len(two):
    print(f"\nCROSS-REGION MODERATION")
    for _, r_ in two.iterrows():
        print(f"  {r_['set']}: {r_['observed']:+.3f}, p = {r_['perm_p']:.4f}, "
              f"q = {r_['q_bh']:.4f}")
    for deg, c in zip((1, 2), position_controlled):
        if c:
            print(f"    A-P controlled (degree {deg}): {c['observed']:+.3f}, "
                  f"p = {c['perm_p']:.4f}")

print(f"\nCAVEATS TO CARRY INTO THE TEXT")
print(f"  - permutation p cannot separate a real effect from a technical gain")
print(f"    difference confounded with group assignment; see 13b and the batch crosstabs")
print(f"  - contrasts flagged 'at resolution limit' cannot go below their floor")
print(f"  - q_powered = 0 for an empty family is not a negative result")
print(f"  - sets clearing the random but not the spatial null show contiguity,")
print(f"    not concentration")
print(f"  - genotype contrasts are CONTROL-POSITIVE; their sign is negated relative")
print(f"    to any run before this version -- check the manuscript text")
print(f"  - the enrichment statistic is subset vs. WHOLE BRAIN; it differs from the")
print(f"    complement form by 1 - k/N and gives an IDENTICAL p, so it is a change")
print(f"    of reported effect size, not a re-analysis")
print(f"  - divisions covering a large share of the brain have no resolution under")
print(f"    either null; they are flagged, not interpreted")

# ---- export ----
tables = {"power_audit": power_audit, "per_mouse_gain": gain_df,
          "gain_explains": gain_explains, "global_breadth": breadth_summary,
          "enrichment": enrichment_summary, "by_major_structure": by_struct,
          "enrichment_by_division": division_summary}
try:
    with pd.ExcelWriter(OUT_DIR / "summary_tables.xlsx") as xl:
        for name, tbl in tables.items():
            tbl.to_excel(xl, sheet_name=name)
    print(f"\nSaved all summary tables -> {OUT_DIR / 'summary_tables.xlsx'}")
except Exception as exc:
    # openpyxl missing, or the file is open in Excel and locked
    print(f"\nExcel export unavailable ({type(exc).__name__}) -- writing CSVs instead:")
    for name, tbl in tables.items():
        path = OUT_DIR / f"summary_{name}.csv"
        tbl.to_csv(path)
        print(f"  {path}")